<a href="https://colab.research.google.com/github/springernature/analytics_engineering_dbt/blob/anup_mane/2026_RM_KPI_report_GBQ_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Important Links

[BRM Report 2026](https://docs.google.com/spreadsheets/d/1e9P_iNih90P07JbFBKxImnxqiXcSRBToufYGhHXfdds/edit?gid=0#gid=0)

[Marketing KPIs Overview](https://docs.google.com/spreadsheets/d/1vsXky3lIZ4Z0ZdX_EDLRBF3qAzqa3MgZ4SbaTvO49II/edit?gid=2062847935#gid=2062847935)

[KPI data sheet 2026](https://docs.google.com/spreadsheets/d/13fBXUHit2jGgbwRSw40vIEC7_g1YukNswo4Gb1u03Vc/edit?gid=1642437922#gid=1642437922)

[Budget sheet 2026](https://docs.google.com/spreadsheets/d/1Kggmjmbu4TWo88vNMf5At0BjimnGorj15akU_q0k3YQ/edit?gid=679258872#gid=679258872)

[Forecast sheet 2026](https://docs.google.com/spreadsheets/d/1Gupk4PAG53NuTXkIj8dM5M8MNAsnXTDHGsDGR2OC3mg/edit?gid=1584856269#gid=1584856269)

#**Authenticate**

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated")

Authenticated


#**Data from GBQ**

###Create KPI tables with individual KPIs

####**ecommerce**
*--ecomRevActMktSLNAT, ecomRevSeoSLNAT, ecomRevActMktSL, ecomRevSeoSL,
 ecomRevActMktSLbooks, ecomRevSeoSLbooks, ecomRevActMktSLeBooks, ecomRevSeoSLeBooks, ecomRevActMktSLppvchap, ecomRevSeoSLppvchap

#####Replace query

In [ ]:
%%bigquery --project  snproject147bce46
--ecomRev, ecomRevActMktSLNAT, ecomRevSeoSLNAT, ecomRevActMktSL, ecomRevSeoSL, ecomRevActMktSLbooksEbks, ecomRevSeoSLbooksEbks, ecomRevActMktSLppvchap, ecomRevSeoSLppvchap, ecomRevActMktSLppvart, ecomRevSeoSLppvart, ecomRevActMktSLsubs, ecomRevSeoSLsubs,
--ecomRevActMktB2BInst, ecomRevSeoB2BInst

CREATE OR REPLACE TABLE `snproject147bce46.2026_kpi_report.ecom`
AS
WITH
  a AS (
    SELECT
      date,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            THEN revenue
          END) AS ecomRevActMktSLNAT,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            THEN revenue
        END) AS ecomRevSeoSLNAT,

  ------------- Remove -------------
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND platform = 'springerlink'
            THEN revenue
          END) AS ecomRevActMktSL,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND platform = 'springerlink'
            THEN revenue
          END) AS ecomRevSeoSL,
  ----------------------------------
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND regexp_contains(item_variant, "print|ebo")
            THEN revenue
          END) AS ecomRevActMktSLbooksEbks,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND regexp_contains(item_variant, "print|ebo")
            THEN revenue
          END) AS ecomRevSeoSLbooksEbks,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND item_variant = "ppv-chapter"
            THEN revenue
          END) AS ecomRevActMktSLppvchap,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND item_variant = "ppv-chapter"
            THEN revenue
          END) AS ecomRevSeoSLppvchap,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND item_variant = "ppv-article"
            THEN revenue
          END) AS ecomRevActMktSLppvart,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND item_variant = "ppv-article"
            THEN revenue
          END) AS ecomRevSeoSLppvart,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND (NOT regexp_contains(item_variant, "ebo|print|ppv|journal|apressAccess") OR item_variant IS NULL)
            THEN revenue
          END) AS ecomRevActMktSLsubs,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND (NOT regexp_contains(item_variant, "ebo|print|ppv|journal|apressAccess") OR item_variant IS NULL)
            THEN revenue
          END) AS ecomRevSeoSLsubs
    FROM
      (
        SELECT
          format_date("%Y-%m", date) AS date,
          CASE
            WHEN active_marketing_incl_coupon_mistrack = TRUE
              THEN "of which: Active Marketing"
            WHEN
              sub_channels LIKE '%seo%'
              AND active_marketing_incl_coupon_mistrack = FALSE
              THEN "of which: SEO"
            ELSE "of which: Non Active Marketing"
            END exec_summary_areas,
          platform,
          item_variant,
          sum(ifnull(item_revenue, 0)) AS revenue
        FROM `snproject-cbb9ca25.ria_ecom_reporting.transactions_full_v2`
        WHERE
          EXTRACT(year FROM date) > 2024
          --     date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
          AND platform
            IS NOT NULL  # this table does not contain natureasia data, we do not need to exclude it
        GROUP BY
          date, active_marketing_incl_coupon_mistrack, sub_channels, platform,
          item_variant
        ORDER BY date ASC
      )
    GROUP BY date
    ORDER BY date ASC
  ),
  b AS (
    WITH
      deduplicated AS (
        SELECT
          * EXCEPT (month, year),
          EXTRACT(MONTH FROM date) AS month,
          EXTRACT(YEAR FROM date) AS year,
          ROW_NUMBER()
            OVER (PARTITION BY transaction_id, item_id ORDER BY Date ASC)
            AS unique_row_number
        FROM `snproject-cbb9ca25.hex_final_tables.b2b_sebs_final`
      )
    SELECT
      FORMAT_DATE("%Y-%m", date) AS date,
      month,
      year,
      SUM(
        CASE
          WHEN revenue_breakdown = '1. of which: Marketing'
            THEN FLOOR(item_revenue)
          END) AS ecomRevActMktB2BInst,
      SUM(
        CASE
          WHEN revenue_breakdown = '3. of which: SEO' THEN FLOOR(item_revenue)
          END) AS ecomRevSeoB2BInst
    FROM
      deduplicated
    WHERE
      date > '2024-12-31'
      AND unique_row_number = 1
    GROUP BY 1, 2, 3
    ORDER BY 1
  )
SELECT
  a.date,
  sum(ecomRevActMktSLNAT) AS ecomRevActMktSLNAT,
  sum(ecomRevSeoSLNAT) AS ecomRevSeoSLNAT,
  sum(ecomRevActMktSL) AS ecomRevActMktSL, -- remove
  sum(ecomRevSeoSL) AS ecomRevSeoSL, -- remove
  sum(ecomRevActMktSLbooksEbks) AS ecomRevActMktSLbooksEbks,
  sum(ecomRevSeoSLbooksEbks) AS ecomRevSeoSLbooksEbks,
  sum(ecomRevActMktSLppvchap) AS ecomRevActMktSLppvchap,
  sum(ecomRevSeoSLppvchap) AS ecomRevSeoSLppvchap,
  sum(ecomRevActMktSLppvart) AS ecomRevActMktSLppvart,
  sum(ecomRevSeoSLppvart) AS ecomRevSeoSLppvart,
  sum(ecomRevActMktSLsubs) AS ecomRevActMktSLsubs,
  sum(ecomRevSeoSLsubs) AS ecomRevSeoSLsubs,
  sum(b.ecomRevActMktB2BInst) AS ecomRevActMktB2BInst,
  sum(b.ecomRevSeoB2BInst) AS ecomRevSeoB2BInst,

  SUM(ecomRevActMktSLNAT)
    + SUM(b.ecomRevActMktB2BInst) AS ecomRevActMkt,  -- (rows 47 and 48 in BRM)

  SUM(ecomRevSeoSLNAT)
    + SUM(b.ecomRevSeoB2BInst) AS ecomRevSeo,

  SUM(ecomRevActMktSLNAT)
    + SUM(b.ecomRevActMktB2BInst)
    + SUM(ecomRevSeoSLNAT)
    + SUM(b.ecomRevSeoB2BInst) AS ecomRev  -- (rows 46 and 50 in BRM)
FROM
  a
FULL JOIN b
  ON a.date = b.date
GROUP BY date
ORDER BY date

Query is running:   0%|          |

""


#####Schedule Query [link](https://console.cloud.google.com/bigquery/scheduled-queries/locations/europe/configs/67bfd0a7-0000-2d78-9c06-d4f547fd8fb4/runs?invt=Abtw2Q&project=snproject147bce46)

In [ ]:
%%bigquery --project  snproject147bce46
--ecomRev, ecomRevActMktSLNAT, ecomRevSeoSLNAT, ecomRevActMktSL, ecomRevSeoSL, ecomRevActMktSLbooksEbks, ecomRevSeoSLbooksEbks, ecomRevActMktSLppvchap, ecomRevSeoSLppvchap, ecomRevActMktSLppvart, ecomRevSeoSLppvart, ecomRevActMktSLsubs, ecomRevSeoSLsubs,
--ecomRevActMktB2BInst, ecomRevSeoB2BInst


INSERT INTO `snproject147bce46.2026_kpi_report.ecom`

WITH
  a AS (
    SELECT
      date,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            THEN revenue
          END) AS ecomRevActMktSLNAT,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            THEN revenue
        END) AS ecomRevSeoSLNAT,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND platform = 'springerlink'
            THEN revenue
          END) AS ecomRevActMktSL,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND platform = 'springerlink'
            THEN revenue
          END) AS ecomRevSeoSL,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND regexp_contains(item_variant, "print|ebo")
            THEN revenue
          END) AS ecomRevActMktSLbooksEbks,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND regexp_contains(item_variant, "print|ebo")
            THEN revenue
          END) AS ecomRevSeoSLbooksEbks,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND item_variant = "ppv-chapter"
            THEN revenue
          END) AS ecomRevActMktSLppvchap,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND item_variant = "ppv-chapter"
            THEN revenue
          END) AS ecomRevSeoSLppvchap,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND item_variant = "ppv-article"
            THEN revenue
          END) AS ecomRevActMktSLppvart,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND item_variant = "ppv-article"
            THEN revenue
          END) AS ecomRevSeoSLppvart,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: Active Marketing"
            AND (NOT regexp_contains(item_variant, "ebo|print|ppv|journal|apressAccess") OR item_variant IS NULL)
            THEN revenue
          END) AS ecomRevActMktSLsubs,
      sum(
        CASE
          WHEN
            exec_summary_areas = "of which: SEO"
            AND (NOT regexp_contains(item_variant, "ebo|print|ppv|journal|apressAccess") OR item_variant IS NULL)
            THEN revenue
          END) AS ecomRevSeoSLsubs
    FROM
      (
        SELECT
          format_date("%Y-%m", date) AS date,
          CASE
            WHEN active_marketing_incl_coupon_mistrack = TRUE
              THEN "of which: Active Marketing"
            WHEN
              sub_channels LIKE '%seo%'
              AND active_marketing_incl_coupon_mistrack = FALSE
              THEN "of which: SEO"
            ELSE "of which: Non Active Marketing"
            END exec_summary_areas,
          platform,
          item_variant,
          sum(ifnull(item_revenue, 0)) AS revenue
        FROM `snproject-cbb9ca25.ria_ecom_reporting.transactions_full_v2`
        WHERE
          date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
          --     date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
          AND platform
            IS NOT NULL  # this table does not contain natureasia data, we do not need to exclude it
        GROUP BY
          date, active_marketing_incl_coupon_mistrack, sub_channels, platform,
          item_variant
        ORDER BY date ASC
      )
    GROUP BY date
    ORDER BY date ASC
  ),
  b AS (
    WITH
      deduplicated AS (
        SELECT
          * EXCEPT (month, year),
          EXTRACT(MONTH FROM date) AS month,
          EXTRACT(YEAR FROM date) AS year,
          ROW_NUMBER()
            OVER (PARTITION BY transaction_id, item_id ORDER BY Date ASC)
            AS unique_row_number
        FROM `snproject-cbb9ca25.hex_final_tables.b2b_sebs_final`
      )
    SELECT
      FORMAT_DATE("%Y-%m", date) AS date,
      month,
      year,
      SUM(
        CASE
          WHEN revenue_breakdown = '1. of which: Marketing'
            THEN FLOOR(item_revenue)
          END) AS ecomRevActMktB2BInst,
      SUM(
        CASE
          WHEN revenue_breakdown = '3. of which: SEO' THEN FLOOR(item_revenue)
          END) AS ecomRevSeoB2BInst
    FROM
      deduplicated
    WHERE
      date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
      AND unique_row_number = 1
    GROUP BY 1, 2, 3
    ORDER BY 1
  )
SELECT
  a.date,
  sum(ecomRevActMktSLNAT) AS ecomRevActMktSLNAT,
  sum(ecomRevSeoSLNAT) AS ecomRevSeoSLNAT,
  sum(ecomRevActMktSL) AS ecomRevActMktSL,
  sum(ecomRevSeoSL) AS ecomRevSeoSL,
  sum(ecomRevActMktSLbooksEbks) AS ecomRevActMktSLbooksEbks,
  sum(ecomRevSeoSLbooksEbks) AS ecomRevSeoSLbooksEbks,
  sum(ecomRevActMktSLppvchap) AS ecomRevActMktSLppvchap,
  sum(ecomRevSeoSLppvchap) AS ecomRevSeoSLppvchap,
  sum(ecomRevActMktSLppvart) AS ecomRevActMktSLppvart,
  sum(ecomRevSeoSLppvart) AS ecomRevSeoSLppvart,
  sum(ecomRevActMktSLsubs) AS ecomRevActMktSLsubs,
  sum(ecomRevSeoSLsubs) AS ecomRevSeoSLsubs,
  sum(b.ecomRevActMktB2BInst) AS ecomRevActMktB2BInst,
  sum(b.ecomRevSeoB2BInst) AS ecomRevSeoB2BInst,

  SUM(ecomRevActMktSLNAT)
    + SUM(b.ecomRevSeoB2BInst) AS ecomRevActMkt,  -- (rows 47 and 48 in BRM)

  SUM(ecomRevSeoSLNAT)
    + SUM(b.ecomRevActMktB2BInst) AS ecomRevSeo,

  # SUM(ecomRevSeoSLbooksEbks) + SUM(ecomRevSeoSLppvchap) + SUM(ecomRevSeoSLppvart) + sum(ecomRevSeoSLsubs) + SUM(b.ecomRevSeoB2BInst) as ecomRevSeo, --(rows 51 to 55 in BRM)

  SUM(ecomRevActMktSLNAT)
    + SUM(b.ecomRevActMktB2BInst)
    + SUM(ecomRevSeoSLbooksEbks)
    + SUM(ecomRevSeoSLppvchap) + SUM(ecomRevSeoSLppvart) + sum(ecomRevSeoSLsubs)
    + SUM(b.ecomRevSeoB2BInst) AS ecomRev  -- (rows 46 and 50 in BRM)
FROM
  a
FULL JOIN b
  ON a.date = b.date
GROUP BY date
ORDER BY date

####researcher

*--elsMkt, elsMktOa, submitClickMkt, submitClickMktOa, JrnlEngMkt,JrnlEngMktOa*

#####Replace query

In [ ]:
%%bigquery --project  snproject147bce46
--elsMkt, elsMktOa, submitClickMkt, submitClickMktOa, JrnlEngMkt,JrnlEngMktOa

CREATE OR REPLACE TABLE `snproject147bce46.2026_kpi_report.researcher`
AS
SELECT
  *
    EXCEPT (
      month,
      year,
      submitClickMkt,
      submitClickMktOa,
      elsMkt,
      elsMktOa,
      JrnlEngMkt,
      JrnlEngMktOa),

  CASE
    WHEN month = 1 AND year = 2025 THEN 210179
    WHEN month = 2 AND year = 2025 THEN 183532
    WHEN month = 3 AND year = 2025 THEN 161988
    WHEN month = 4 AND year = 2025 THEN 168217
    WHEN month = 5 AND year = 2025 THEN 166544
    WHEN month = 6 AND year = 2025 THEN 154848
    WHEN month = 7 AND year = 2025 THEN 173618
    WHEN month = 8 AND year = 2025 THEN 156436
    WHEN month = 9 AND year = 2025 THEN 204383
    WHEN month = 10 AND year = 2025 THEN 263082
    WHEN month = 11 AND year = 2025 THEN 240396
    WHEN month = 12 AND year = 2025 THEN 193206
    ELSE submitClickMkt
    END AS submitClickMkt,
  CASE
    WHEN month = 1 AND year = 2025 THEN 131491
    WHEN month = 2 AND year = 2025 THEN 111716
    WHEN month = 3 AND year = 2025 THEN 84695
    WHEN month = 4 AND year = 2025 THEN 89630
    WHEN month = 5 AND year = 2025 THEN 89621
    WHEN month = 6 AND year = 2025 THEN 81840
    WHEN month = 7 AND year = 2025 THEN 89162
    WHEN month = 8 AND year = 2025 THEN 77174
    WHEN month = 9 AND year = 2025 THEN 81706
    WHEN month = 10 AND year = 2025 THEN 94083
    WHEN month = 11 AND year = 2025 THEN 102969
    WHEN month = 12 AND year = 2025 THEN 76014
    ELSE submitClickMktOa
    END AS submitClickMktOa,
  CASE
    WHEN month = 1 AND year = 2025 THEN 38343
    WHEN month = 2 AND year = 2025 THEN 43333
    WHEN month = 3 AND year = 2025 THEN 47328
    WHEN month = 4 AND year = 2025 THEN 48267
    WHEN month = 5 AND year = 2025 THEN 23940
    WHEN month = 6 AND year = 2025 THEN 18760
    WHEN month = 7 AND year = 2025 THEN 24052
    WHEN month = 8 AND year = 2025 THEN 19652
    WHEN month = 9 AND year = 2025 THEN 22133
    WHEN month = 10 AND year = 2025 THEN 18791
    WHEN month = 11 AND year = 2025 THEN 18469
    WHEN month = 12 AND year = 2025 THEN 13485
    ELSE elsMkt
    END AS elsMkt,
  CASE
    WHEN month = 1 AND year = 2025 THEN 22315
    WHEN month = 2 AND year = 2025 THEN 26523
    WHEN month = 3 AND year = 2025 THEN 27294
    WHEN month = 4 AND year = 2025 THEN 28058
    WHEN month = 5 AND year = 2025 THEN 14106
    WHEN month = 6 AND year = 2025 THEN 11690
    WHEN month = 7 AND year = 2025 THEN 14899
    WHEN month = 8 AND year = 2025 THEN 12370
    WHEN month = 9 AND year = 2025 THEN 13445
    WHEN month = 10 AND year = 2025 THEN 11132
    WHEN month = 11 AND year = 2025 THEN 10589
    WHEN month = 12 AND year = 2025 THEN 6532
    ELSE elsMktOa
    END AS elsMktOa,
  ##journal_engagement event was modified to fire when user reaches 50% scroll on journal pages
  ##The event was modified on 2025-04-16
  CASE
    WHEN month = 1 AND year = 2025 THEN 0
    WHEN month = 2 AND year = 2025 THEN 0
    WHEN month = 3 AND year = 2025 THEN 0
    WHEN month = 4 AND year = 2025 THEN 0
    WHEN month = 5 AND year = 2025 THEN 0
    WHEN month = 6 AND year = 2025 THEN 288650
    WHEN month = 7 AND year = 2025 THEN 371331
    WHEN month = 8 AND year = 2025 THEN 348942
    WHEN month = 9 AND year = 2025 THEN 437914
    WHEN month = 10 AND year = 2025 THEN 366395
    WHEN month = 11 AND year = 2025 THEN 341641
    WHEN month = 12 AND year = 2025 THEN 303013
    ELSE JrnlEngMkt
    END AS JrnlEngMkt,
  CASE
    WHEN month = 1 AND year = 2025 THEN 0
    WHEN month = 2 AND year = 2025 THEN 0
    WHEN month = 3 AND year = 2025 THEN 0
    WHEN month = 4 AND year = 2025 THEN 0
    WHEN month = 5 AND year = 2025 THEN 0
    WHEN month = 6 AND year = 2025 THEN 117870
    WHEN month = 7 AND year = 2025 THEN 169535
    WHEN month = 8 AND year = 2025 THEN 176417
    WHEN month = 9 AND year = 2025 THEN 196448
    WHEN month = 10 AND year = 2025 THEN 148865
    WHEN month = 11 AND year = 2025 THEN 134304
    WHEN month = 12 AND year = 2025 THEN 115781
    ELSE JrnlEngMktOa
    END AS JrnlEngMktOa
FROM
  (
    SELECT
      date,
      month,
      year,
      SUM(submitclickmkt) AS submitClickMkt,
      IFNULL(SUM(submitclickmktoa), 0.0) AS submitClickMktOa,
      SUM(elsMkt) AS elsMkt,
      IFNULL(SUM(elsMktOa), 0.0) AS elsMktOa,
      SUM(JrnlEngMkt) AS JrnlEngMkt,
      IFNULL(SUM(JrnlEngMktOa), 0.0) AS JrnlEngMktOa
    FROM
      (
        SELECT
          FORMAT_DATE("%Y-%m", date) AS date,
          EXTRACT(month FROM date) AS month,
          EXTRACT(year FROM date) AS year,
          CASE
            WHEN research_marketing_activity IS TRUE
              THEN CAST(SUM(kpi_submit_click) AS FLOAT64)
            END AS submitClickMkt,
          CASE
            WHEN research_marketing_activity IS TRUE
              THEN CAST(SUM(els) AS FLOAT64)
            END AS elsMkt,
          CASE
            WHEN research_marketing_activity IS TRUE
              THEN CAST(SUM(journal_engagement) AS FLOAT64)
            END AS JrnlEngMkt,
          CASE
            WHEN oatype = "OA" AND research_marketing_activity IS TRUE
              THEN CAST(Sum(kpi_submit_click) AS FLOAT64)
            END AS submitClickMktOa,
          CASE
            WHEN oatype = "OA" AND research_marketing_activity IS TRUE
              THEN CAST(SUM(els) AS FLOAT64)
            END AS elsMktOa,
          CASE
            WHEN oatype = "OA" AND research_marketing_activity IS TRUE
              THEN CAST(SUM(journal_engagement) AS FLOAT64)
            END AS JrnlEngMktOa,
        FROM `snproject-cbb9ca25.ga4_ria_researcher_report_dataform.j7`
        WHERE EXTRACT(year FROM date) > 2024
        # add clause here for strict date selection to insert only past month's data into report
        GROUP BY date, research_marketing_activity, oatype, month, year
        ORDER BY date ASC
      )
    GROUP BY date, month, year
  )
ORDER BY date ASC

Query is running:   0%|          |

""


#####Schedule Query [link](https://console.cloud.google.com/bigquery/scheduled-queries/locations/europe/configs/67ae0071-0000-2a1e-9765-c82add78c1cc/runs?invt=Abtw2Q&project=snproject147bce46)

In [ ]:
%%bigquery --project  snproject147bce46
--elsMkt, elsMktOa, submitClickMkt, submitClickMktOa, JrnlEngMkt,JrnlEngMktOa

INSERT INTO `snproject147bce46.2026_kpi_report.researcher`
SELECT
  date,
  SUM(submitclickmkt) AS submitClickMkt,
  IFNULL(SUM(submitclickmktoa), 0.0) AS submitClickMktOa,
  SUM(elsMkt) AS elsMkt,
  IFNULL(SUM(elsMktOa), 0.0) AS elsMktOa,
  SUM(JrnlEngMkt) AS JrnlEngMkt,
  IFNULL(SUM(JrnlEngMktOa), 0.0) AS JrnlEngMktOa
FROM
  (
    SELECT
      FORMAT_DATE("%Y-%m", date) AS date,
      EXTRACT(month FROM date) AS month,
      EXTRACT(year FROM date) AS year,
      CASE
        WHEN research_marketing_activity IS TRUE
          THEN CAST(SUM(kpi_submit_click) AS FLOAT64)
        END AS submitClickMkt,
      CASE
        WHEN research_marketing_activity IS TRUE
          THEN CAST(SUM(els) AS FLOAT64)
        END AS elsMkt,
      CASE
        WHEN research_marketing_activity IS TRUE
          THEN CAST(SUM(journal_engagement) AS FLOAT64)
        END AS JrnlEngMkt,
      CASE
        WHEN oatype = "OA" AND research_marketing_activity IS TRUE
          THEN CAST(Sum(kpi_submit_click) AS FLOAT64)
        END AS submitClickMktOa,
      CASE
        WHEN oatype = "OA" AND research_marketing_activity IS TRUE
          THEN CAST(SUM(els) AS FLOAT64)
        END AS elsMktOa,
      CASE
        WHEN oatype = "OA" AND research_marketing_activity IS TRUE
          THEN CAST(SUM(journal_engagement) AS FLOAT64)
        END AS JrnlEngMktOa,
    FROM `snproject-cbb9ca25.ga4_ria_researcher_report_dataform.j7`
    WHERE
      date
      BETWEEN date_trunc(date_sub(current_date(), INTERVAL 1 month), month)
      AND last_day(date_sub(current_date(), INTERVAL 1 month), month)
    # add clause here for strict date selection to insert only past month's data into report
    GROUP BY date, research_marketing_activity, oatype, month, year
    ORDER BY date ASC
  )
GROUP BY date

Query is running:   0%|          |

Downloading:   0%|          |

,date,elsMkt,elsMktOa,submitClickMkt,submitClickMktOa,JrnlEngMkt,JrnlEngMktOa
0,2025-03,47234.0,27242.0,161783.0,84579.0,1199540.0,481584.0


####SEO - Traditional and AI SEO code

######Replace Query

In [ ]:
%%bigquery --project  snproject147bce46
-- seo, AIseo, Traditionalseo


CREATE OR REPLACE TABLE `snproject147bce46.2026_kpi_report.seo`
AS
SELECT
  * EXCEPT (month, year, seo, AIseo, Traditionalseo),
  CASE
    WHEN month = 1 AND year = 2025 THEN 43634657
    WHEN month = 2 AND year = 2025 THEN 44749677
    WHEN month = 3 AND year = 2025 THEN 51480414
    WHEN month = 4 AND year = 2025 THEN 48534802
    WHEN month = 5 AND year = 2025 THEN 47544336
    WHEN month = 6 AND year = 2025 THEN 41265145
    WHEN month = 7 AND year = 2025 THEN 41286131
    WHEN month = 8 AND year = 2025 THEN 38322101
    WHEN month = 9 AND year = 2025 THEN 46167624
    WHEN month = 10 AND year = 2025 THEN 49742351
    WHEN month = 11 AND year = 2025 THEN 49321207
    WHEN month = 12 AND year = 2025 THEN 45957198
    ELSE seo
    END AS seo,
  CASE
    WHEN month = 1 AND year = 2025 THEN 0
    WHEN month = 2 AND year = 2025 THEN 0
    WHEN month = 3 AND year = 2025 THEN 0
    WHEN month = 4 AND year = 2025 THEN 0
    WHEN month = 5 AND year = 2025 THEN 0
    WHEN month = 6 AND year = 2025 THEN 0
    WHEN month = 7 AND year = 2025 THEN 0
    WHEN month = 8 AND year = 2025 THEN 4971702
    WHEN month = 9 AND year = 2025 THEN 7262138
    WHEN month = 10 AND year = 2025 THEN 8677816
    WHEN month = 11 AND year = 2025 THEN 8933931
    WHEN month = 12 AND year = 2025 THEN 7225869
    ELSE AIseo
    END AS AIseo,
  CASE
    WHEN month = 1 AND year = 2025 THEN 0
    WHEN month = 2 AND year = 2025 THEN 0
    WHEN month = 3 AND year = 2025 THEN 0
    WHEN month = 4 AND year = 2025 THEN 0
    WHEN month = 5 AND year = 2025 THEN 0
    WHEN month = 6 AND year = 2025 THEN 0
    WHEN month = 7 AND year = 2025 THEN 0
    WHEN month = 8 AND year = 2025 THEN 33350399
    WHEN month = 9 AND year = 2025 THEN 38905486
    WHEN month = 10 AND year = 2025 THEN 41064535
    WHEN month = 11 AND year = 2025 THEN 40387276
    WHEN month = 12 AND year = 2025 THEN 38731329
    ELSE Traditionalseo
    END AS Traditionalseo
FROM
  (
    SELECT
      format_date("%Y-%m", date) AS date,
      EXTRACT(month FROM date) AS month,
      EXTRACT(year FROM date) AS year,
      SUM(CASE WHEN main_channels = 'seo' THEN sessions END) AS seo,
      SUM(CASE WHEN sub_channels = 'ai seo' THEN sessions END) AS AIseo,
      SUM(CASE WHEN sub_channels = 'traditional seo' THEN sessions END)
        AS Traditionalseo
    FROM `snproject-cbb9ca25.ga4_ria_researcher_report_dataform.j7`
    WHERE
      EXTRACT(year FROM date) > 2024
      AND imprint IS NOT NULL
    GROUP BY 1, 2, 3
  )
GROUP BY date, month, year, seo, AIseo, Traditionalseo
ORDER BY date


######Scheduled Query

In [ ]:
%%bigquery --project  snproject147bce46
-- seo, AIseo, Traditionalseo
INSERT INTO `snproject147bce46.2026_kpi_report.seo`
SELECT
  format_date("%Y-%m", date) AS date,
  SUM(CASE WHEN main_channels = 'seo' THEN sessions END) AS seo,
  SUM(CASE WHEN sub_channels = 'ai seo' THEN sessions END) AS AIseo,
  SUM(CASE WHEN sub_channels = 'traditional seo' THEN sessions END) AS Traditionalseo
FROM `snproject-cbb9ca25.ga4_ria_researcher_report_dataform.j7`
WHERE
  date
    BETWEEN date_trunc(date_sub(current_date(), INTERVAL 1 month), month)
    AND last_day(date_sub(current_date(), INTERVAL 1 month), month)
  AND imprint IS NOT NULL
GROUP BY 1



####**Submissions**
*--submMktGen, submMktGenOa, submMktInfl1Plus, submMktInflCol1Plus, submMktInfl3Plus, submMktInfl3PlusOa, powerAuthors*

#####Replace query

In [ ]:
%%bigquery --project  snproject147bce46
--submMktGen, submMktGenOa, submMktInfl1Plus, submMktInflCol1Plus, submMktInfl3Plus, submMktInfl3PlusOa, powerAuthors

CREATE OR REPLACE TABLE `snproject147bce46.2026_kpi_report.submissions`
AS
WITH
  mis AS (
    SELECT
      format_date("%Y-%m", month_start_date) AS date,
      sum(mkt_infl_1plus) AS submMktInfl1Plus,
      sum(mkt_infl_1plus_collections) AS submMktInflCol1Plus,
      sum(mkt_infl_3plus) AS submMktInfl3Plus,
      sum(mkt_infl_3plus_oa) AS submMktInfl3PlusOa,
      sum(mkt_inf_3plus_otr) AS submMktInfl3PlusOtr
    FROM `snproject-cbb9ca25.RM_Submissions_mktg_infl.brm_subs_reporting_2026`
    WHERE EXTRACT(year FROM month_start_date) > 2024
    GROUP BY 1
  ),
  subs AS (
    SELECT
      format_date("%Y-%m", month_start_date) AS date,
      0 AS submMktGen,
      0 AS submMktGenOa
    FROM `snproject-cbb9ca25.RM_Submissions_reporting.brm_subs_reporting_2025`
    WHERE EXTRACT(year FROM month_start_date) > 2024
    GROUP BY 1
  ),
  pwrauthors AS (
    SELECT
      format_date("%Y-%m", month_start_date) AS date,
      sum(subs_with_power_authors) AS powerAuthors
    FROM
      `snproject-cbb9ca25.RM_Submissions_reporting.monthly_power_authors_feb25_2`
    WHERE EXTRACT(year FROM month_start_date) > 2024
    GROUP BY 1
  )
SELECT
  * EXCEPT (submMktGen, submMktGenOa),
  ifnull(submMktGen, 0.0) AS submMktGen,
  ifnull(submMktGenOa, 0.0) AS submMktGenOa
FROM
  mis
FULL JOIN
  subs
  USING (date)
FULL JOIN
  pwrauthors
  USING (date)
ORDER BY date

Query is running:   0%|          |

""


#####Schedule Query [link](https://console.cloud.google.com/bigquery/scheduled-queries/locations/europe/configs/67aea34d-0000-20ab-aece-001a114d304c/runs?invt=AbtxBg&project=snproject147bce46)

In [ ]:
%%bigquery --project  snproject147bce46
--submMktGen, submMktGenOa, submMktInfl1Plus, submMktInflCol1Plus, submMktInfl3Plus, submMktInfl3PlusOa, powerAuthors

insert into `snproject147bce46.2026_kpi_report.submissions`

WITH
  mis AS (
    SELECT
      format_date("%Y-%m", month_start_date) AS date,
      sum(mkt_infl_1plus) AS submMktInfl1Plus,
      sum(mkt_infl_1plus_collections) AS submMktInflCol1Plus,
      sum(mkt_infl_3plus) AS submMktInfl3Plus,
      sum(mkt_infl_3plus_oa) AS submMktInfl3PlusOa,
      sum(mkt_inf_3plus_otr) AS submMktInfl3PlusOtr
    FROM `snproject-cbb9ca25.RM_Submissions_mktg_infl.brm_subs_reporting_2026`
    where month_start_date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
    GROUP BY 1
  ),
  subs AS (
    SELECT
      format_date("%Y-%m", month_start_date) AS date,
      0 AS submMktGen,
      0 AS submMktGenOa
    FROM `snproject-cbb9ca25.RM_Submissions_reporting.brm_subs_reporting_2025`
    where month_start_date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
    GROUP BY 1
  ),
  pwrauthors AS (
    SELECT
      format_date("%Y-%m", month_start_date) AS date,
      sum(subs_with_power_authors) AS powerAuthors
    FROM
      `snproject-cbb9ca25.RM_Submissions_reporting.monthly_power_authors_feb25_2`
    where month_start_date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
    GROUP BY 1
  )
SELECT
  * EXCEPT (submMktGen, submMktGenOa),
  ifnull(submMktGen, 0.0) AS submMktGen,
  ifnull(submMktGenOa, 0.0) AS submMktGenOa
FROM
  mis
FULL JOIN
  subs
  USING (date)
FULL JOIN
  pwrauthors
  USING (date)
ORDER BY date

####**Book Leads**

#####Replace query

In [ ]:
%%bigquery --project  snproject147bce46
--bookLeads, bookLeadsAcc, bookCntctSgnMkt

CREATE OR REPLACE TABLE `snproject147bce46.2026_kpi_report.books`
AS
SELECT
  * EXCEPT (month, year, bookLeads, bookCntctSgnMkt),
  CASE
    WHEN month = 1 AND year = 2025 THEN 1416
    WHEN month = 2 AND year = 2025 THEN 1367
    WHEN month = 3 AND year = 2025 THEN 1355
    WHEN month = 4 AND year = 2025 THEN 1447
    WHEN month = 5 AND year = 2025 THEN 1539
    WHEN month = 6 AND year = 2025 THEN 1709
    WHEN month = 7 AND year = 2025 THEN 1875
    WHEN month = 8 AND year = 2025 THEN 1746
    WHEN month = 9 AND year = 2025 THEN 1438
    WHEN month = 10 AND year = 2025 THEN 1331
    WHEN month = 11 AND year = 2025 THEN 1279
    WHEN month = 12 AND year = 2025 THEN 1515
    ELSE bookLeads
    END AS bookLeads,
  CASE
    WHEN month = 1 AND year = 2025 THEN 43
    WHEN month = 2 AND year = 2025 THEN 78
    WHEN month = 3 AND year = 2025 THEN 114
    WHEN month = 4 AND year = 2025 THEN 71
    WHEN month = 5 AND year = 2025 THEN 111
    WHEN month = 6 AND year = 2025 THEN 93
    WHEN month = 7 AND year = 2025 THEN 145
    WHEN month = 8 AND year = 2025 THEN 113
    WHEN month = 9 AND year = 2025 THEN 84
    WHEN month = 10 AND year = 2025 THEN 130
    WHEN month = 11 AND year = 2025 THEN 115
    WHEN month = 12 AND year = 2025 THEN 64
    ELSE bookCntctSgnMkt
    END AS bookCntctSgnMkt,
FROM
  (
    WITH
      a AS (
        SELECT
          format_date('%Y-%m', SubmitDate) AS date,
          EXTRACT(month FROM SubmitDate) AS month,
          EXTRACT(year FROM SubmitDate) AS year,
          COUNT(DISTINCT EditionId) AS bookLeads,
        FROM `snproject-cbb9ca25.ga4_ria_researcher_report.BooksMarketing`
        WHERE EXTRACT(year FROM SubmitDate) > 2024
        GROUP BY 1, 2, 3
      ),
      b AS (
        SELECT
          format_date('%Y-%m', ContractSigned) AS date,
          EXTRACT(month FROM ContractSigned) AS month,
          EXTRACT(year FROM ContractSigned) AS year,
          COUNT(DISTINCT ContractSigned) AS bookCntctSgnMkt,
        FROM `snproject-cbb9ca25.ga4_ria_researcher_report.BooksMarketing`
        WHERE EXTRACT(year FROM ContractSigned) > 2024
        GROUP BY 1, 2, 3
      ),
      c AS (
        SELECT
          format_date("%Y-%m", statusdate) AS date,
          EXTRACT(month FROM statusdate) AS month,
          EXTRACT(year FROM statusdate) AS year,
          COUNT(EditionId) AS bookLeadsAcc
        FROM
          (
            SELECT DISTINCT
              EditionId,
              ProposedTitle,
              status,
              StatusHistory.StatusUpdateDate[OFFSET(pos)] AS Statusdate
            FROM `snproject-cbb9ca25.ga4_ria_researcher_report.BooksMarketing`
            CROSS JOIN UNNEST(StatusHistory.status) AS status WITH OFFSET pos
            WHERE status = "Submission_to_MyBookSubmission_succeeded"
          )
        WHERE EXTRACT(year FROM statusdate) > 2024
        GROUP BY date, month, year
      )
    SELECT
      a.date,
      a.month,
      a.year,
      sum(a.bookLeads) AS bookLeads,
      sum(c.bookLeadsAcc) AS bookLeadsAcc,
      sum(b.bookCntctSgnMkt) AS bookCntctSgnMkt
    FROM a
    LEFT JOIN b
      USING (date)
    LEFT JOIN c
      USING (date)
    GROUP BY date, month, year
  )
ORDER BY date

#####Schedule Query [link](https://console.cloud.google.com/bigquery/scheduled-queries/locations/europe/configs/67aa71f2-0000-2be3-b100-001a114cf034/runs?invt=AbtxCA&project=snproject147bce46)

In [ ]:
%%bigquery --project  snproject147bce46
--bookLeads, bookLeadsAcc, bookCntctSgnMkt

INSERT INTO `snproject147bce46.2026_kpi_report.books`
WITH
  a AS (
    SELECT
      format_date('%Y-%m', SubmitDate) AS date,
      COUNT(DISTINCT EditionId) AS bookLeads,
    FROM `snproject-cbb9ca25.ga4_ria_researcher_report.BooksMarketing`
    WHERE
      EXTRACT(date FROM SubmitDate)
      BETWEEN date_trunc(date_sub(current_date(), INTERVAL 1 month), month)
      AND last_day(date_sub(current_date(), INTERVAL 1 month), month)
    GROUP BY 1
  ),
  b AS (
    SELECT
      format_date('%Y-%m', ContractSigned) AS date,
      COUNT(DISTINCT ContractSigned) AS bookCntctSgnMkt,
    FROM `snproject-cbb9ca25.ga4_ria_researcher_report.BooksMarketing`
    WHERE
      EXTRACT(date FROM ContractSigned)
      BETWEEN date_trunc(date_sub(current_date(), INTERVAL 1 month), month)
      AND last_day(date_sub(current_date(), INTERVAL 1 month), month)
    GROUP BY 1
  ),
  c AS (
    SELECT
      format_date("%Y-%m", Statusdate) AS date,
      COUNT(EditionId) AS bookLeadsAcc
    FROM
      (
        SELECT DISTINCT
          EditionId,
          ProposedTitle,
          status,
          StatusHistory.StatusUpdateDate[OFFSET(pos)] AS Statusdate
        FROM `snproject-cbb9ca25.ga4_ria_researcher_report.BooksMarketing`
        CROSS JOIN UNNEST(StatusHistory.status) AS status WITH OFFSET pos
        WHERE status = "Submission_to_MyBookSubmission_succeeded"
      )
    WHERE
      Statusdate
      BETWEEN date_trunc(date_sub(current_date(), INTERVAL 1 month), month)
      AND last_day(date_sub(current_date(), INTERVAL 1 month), month)
    GROUP BY date
  )
SELECT
  a.date,
  sum(a.bookLeads) AS bookLeads,
  sum(c.bookLeadsAcc) AS bookLeadsAcc,
  sum(b.bookCntctSgnMkt) AS bookCntctSgnMkt
FROM a
LEFT JOIN b
  USING (date)
LEFT JOIN c
  USING (date)
GROUP BY date
ORDER BY date


####**MQL Leads and SALs**

#####Replace query

In [ ]:
%%bigquery --project  snproject147bce46
--salJrnl; salBks; salDas; mqlJrnl; mqlBks; mqlDas

CREATE OR REPLACE TABLE `snproject147bce46.2026_kpi_report.mql_jbd`
AS
WITH
  current_totals AS (
    SELECT
      FORMAT_DATE("%Y-%m", CAST(FORMAT_DATETIME("%F", created_date) AS date)) AS date,
      SUM(CASE WHEN high_level_product = 'Journals' THEN total_sal END) AS salJrnl,
      SUM(CASE WHEN high_level_product = 'eBooks' THEN total_sal END) AS salBks,
      SUM(CASE WHEN high_level_product = 'Databases' THEN total_sal END) AS salDas,
      SUM(CASE WHEN high_level_product = 'Journals' THEN total_mql END) AS mqlJrnl,
      SUM(CASE WHEN high_level_product = 'eBooks' THEN total_mql END) AS mqlBks,
      SUM(CASE WHEN high_level_product = 'Databases' THEN total_mql END) AS mqlDas
    FROM `snproject-cbb9ca25.hex_final_tables.b2b_leads_all`
    WHERE created_year > 2024
    GROUP BY date
  ),
  ytd AS (
    SELECT
      date,
      SUM(salJrnl)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS salJrnl_ytd,
      SUM(salBks)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS salBks_ytd,
      SUM(salDas)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS salDas_ytd,
      SUM(mqlJrnl)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS mqlJrnl_ytd,
      SUM(mqlBks)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS mqlBks_ytd,
      SUM(mqlDas)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS mqlDas_ytd
    FROM current_totals
  ),
  final
  AS (
    SELECT
      date,
      salJrnl_ytd - COALESCE(LAG(salJrnl_ytd) OVER (ORDER BY date),0) AS salJrnl,
      salBks_ytd - COALESCE(LAG(salBks_ytd) OVER (ORDER BY date),0) AS salBks,
      salDas_ytd - COALESCE(LAG(salDas_ytd) OVER (ORDER BY date),0) AS salDas,
      mqlJrnl_ytd - COALESCE(LAG(mqlJrnl_ytd) OVER (ORDER BY date),0) AS mqlJrnl,
      mqlBks_ytd - COALESCE(LAG(mqlBks_ytd) OVER (ORDER BY date),0) AS mqlBks,
      mqlDas_ytd - COALESCE(LAG(mqlDas_ytd) OVER (ORDER BY date),0) AS mqlDas,
    FROM ytd
    ORDER BY date
  )
SELECT *
FROM final

#####Schedule Query [link](https://console.cloud.google.com/bigquery/scheduled-queries/locations/europe/configs/67c77d74-0000-2d76-aa64-94eb2c0dcbb6/runs?invt=AbtxCg&project=snproject147bce46)

In [ ]:
%%bigquery --project  snproject147bce46
--salJrnl; salBks; salDas; mql

INSERT INTO `snproject147bce46.2026_kpi_report.mql_jbd`

WITH
  current_totals AS (
    SELECT
      FORMAT_DATE("%Y-%m", CAST(FORMAT_DATETIME("%F", created_date) AS date)) AS date,
      SUM(CASE WHEN high_level_product = 'Journals' THEN total_sal END) AS salJrnl,
      SUM(CASE WHEN high_level_product = 'eBooks' THEN total_sal END) AS salBks,
      SUM(CASE WHEN high_level_product = 'Databases' THEN total_sal END) AS salDas,
      SUM(CASE WHEN high_level_product = 'Journals' THEN total_mql END) AS mqlJrnl,
      SUM(CASE WHEN high_level_product = 'eBooks' THEN total_mql END) AS mqlBks,
      SUM(CASE WHEN high_level_product = 'Databases' THEN total_mql END) AS mqlDas
    FROM `snproject-cbb9ca25.hex_final_tables.b2b_leads_all`
    WHERE created_year > 2024
    GROUP BY date
  ),
  previous_totals AS (
    SELECT
      date,
      SUM(salJrnl_ytd) AS prev_salJrnl,
      SUM(salBks_ytd) AS prev_salBks,
      SUM(salDas_ytd) AS prev_salDas,
      SUM(mqlJrnl_ytd) AS prev_mqlJrnl,
      SUM(mqlBks_ytd) AS prev_mqlBks,
      SUM(mqlDas_ytd) AS prev_mqlDas
    FROM `snproject147bce46.2026_kpi_report.ytd_mql_jbd`
    GROUP BY date
  ),
  ytd AS (
    SELECT
      date,
      SUM(salJrnl)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS salJrnl_ytd,
      SUM(salBks)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS salBks_ytd,
      SUM(salDas)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS salDas_ytd,
      SUM(mqlJrnl)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS mqlJrnl_ytd,
      SUM(mqlBks)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS mqlBks_ytd,
      SUM(mqlDas)
        OVER (
          ORDER BY date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS mqlDas_ytd
    FROM current_totals
  ),
  final
  AS (
    SELECT
      ct.date,
      ct.salJrnl_ytd - COALESCE(LAG(pt.prev_salJrnl) OVER (ORDER BY pt.date),0) AS salJrnl,
      ct.salBks_ytd - COALESCE(LAG(pt.prev_salBks) OVER (ORDER BY pt.date),0) AS salBks,
      ct.salDas_ytd - COALESCE(LAG(pt.prev_salDas) OVER (ORDER BY pt.date),0) AS salDas,
      ct.mqlJrnl_ytd - COALESCE(LAG(pt.prev_mqlJrnl) OVER (ORDER BY pt.date),0) AS mqlJrnl,
      ct.mqlBks_ytd - COALESCE(LAG(pt.prev_mqlBks) OVER (ORDER BY pt.date),0) AS mqlBks,
      ct.mqlDas_ytd - COALESCE(LAG(pt.prev_mqlDas) OVER (ORDER BY pt.date),0) AS mqlDas,
    FROM ytd ct
    LEFT JOIN previous_totals pt
      USING (date)
    ORDER BY date
  )
SELECT *
FROM final
WHERE PARSE_DATE("%Y-%m", date) between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)

####**MQL Rev New Logic**

#####Replace query

In [ ]:
%%bigquery --project  snproject147bce46

CREATE OR REPLACE TABLE `snproject147bce46.2026_kpi_report.mqlRev_jbd`
AS
WITH
  b2b_sf_con_rev AS (
    SELECT
      "b2b_sf_con_rev" AS revenue_type,
      FORMAT_DATE("%Y-%m", opp_close_date) AS date,
      line_item_high_level_product,
      line_item_type_of_business,
      ROUNd(SUM(line_item_total_price_conv), 2) AS revenue
    FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_con_rev_final_v2`
    WHERE opp_close_date > '2024-12-31'
    GROUP BY 1, 2, 3, 4
  ),
  b2b_sf_non_sql_rev AS (
    SELECT
      "b2b_sf_non_sql_rev" AS revenue_type,
      FORMAT_DATE("%Y-%m", opp_close_date) AS date,
      line_item_high_level_product,
      line_item_type_of_business,
      ROUND(SUM(line_item_total_price_conv), 2) AS revenue
    FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_non_sql_rev_final_v2`
    WHERE opp_close_date > '2024-12-31'
    GROUP BY 1, 2, 3, 4
  ),
  b2b_sf_sql_rev AS (
    SELECT
      "b2b_sf_sql_rev" AS revenue_type,
      FORMAT_DATE("%Y-%m", opp_close_date) AS date,
      line_item_high_level_product,
      line_item_type_of_business,
      ROUND(SUM(line_item_total_price_conv), 2) AS revenue
    FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_sql_rev_final_v2`
    WHERE opp_close_date > '2024-12-31'
    GROUP BY 1, 2, 3, 4
  ),
  combined_data AS (
    SELECT * FROM b2b_sf_con_rev
    UNION ALL
    SELECT * FROM b2b_sf_sql_rev
    UNION ALL
    SELECT * FROM b2b_sf_non_sql_rev
  ),
  individual_kpis AS (
    SELECT
      date,
      -- --Marketing lead influenced revenue
      -- SUM(CASE WHEN line_item_type_of_business = "New" OR line_item_type_of_business IS NULL THEN revenue ELSE 0 END) AS mqlRevNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product != 'Journals'
            AND line_item_type_of_business
              IN ("Renewal", "Win Back", "Recurring")
            THEN revenue
          ELSE 0
          END) AS mqlRevRenewal,
      SUM(
        CASE WHEN line_item_type_of_business = "Upsell" THEN revenue ELSE 0 END)
        AS mqlRevUpsell,
      -- Individual B2B KPIs
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Journals'
            AND (
              line_item_type_of_business = "New"
              OR line_item_type_of_business IS NULL)
            THEN revenue
          ELSE 0
          END) AS mqlRevJrnlNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Journals'
            AND line_item_type_of_business = 'Upsell'
            THEN revenue
          ELSE 0
          END) AS mqlRevJrnlUpsell,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'eBooks'
            AND (
              line_item_type_of_business = "New"
              OR line_item_type_of_business IS NULL)
            THEN revenue
          ELSE 0
          END) AS mqlRevBksNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'eBooks'
            AND line_item_type_of_business = 'Upsell'
            THEN revenue
          ELSE 0
          END) AS mqlRevBksUpsell,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'eBooks'
            AND (
              line_item_type_of_business
              IN ("Renewal", "Win Back", "Recurring"))
            THEN revenue
          ELSE 0
          END) AS mqlRevBksRenewal,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Databases'
            AND (
              line_item_type_of_business = "New"
              OR line_item_type_of_business IS NULL)
            THEN revenue
          ELSE 0
          END) AS mqlRevDasNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Databases'
            AND line_item_type_of_business = 'Upsell'
            THEN revenue
          ELSE 0
          END) AS mqlRevDasUpsell,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Databases'
            AND (
              line_item_type_of_business
              IN ("Renewal", "Win Back", "Recurring"))
            THEN revenue
          ELSE 0
          END) AS mqlRevDasRenewal,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Other'
            AND (
              line_item_type_of_business = "New"
              OR line_item_type_of_business IS NULL)
            THEN revenue
          ELSE 0
          END) AS mqlRevOtherNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Other'
            AND line_item_type_of_business = 'Upsell'
            THEN revenue
          ELSE 0
          END) AS mqlRevOtherUpsell,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Other'
            AND (
              line_item_type_of_business
              IN ("Renewal", "Win Back", "Recurring"))
            THEN revenue
          ELSE 0
          END) AS mqlRevOtherRenewal
    FROM combined_data
    GROUP BY 1
  )
SELECT
  *,
  -- Combining individual B2B KPIs
  mqlRevJrnlNew + mqlRevJrnlUpsell AS mqlRevJrnl,
  mqlRevBksNew + mqlRevBksUpsell + mqlRevBksRenewal AS mqlRevBks,
  mqlRevDasNew + mqlRevDasUpsell + mqlRevDasRenewal AS mqlRevDas,
  mqlRevOtherNew + mqlRevOtherUpsell + mqlRevOtherRenewal AS mqlRevOther
FROM individual_kpis
ORDER BY 1 ASC

#####Schedule Query [link](https://console.cloud.google.com/bigquery/scheduled-queries/locations/europe/configs/67d627df-0000-2268-afb2-001a114c211a/runs?invt=AbtxCw&project=snproject147bce46)

In [ ]:
%%bigquery --project  snproject147bce46

INSERT INTO `snproject147bce46.2026_kpi_report.mqlRev_jbd`
WITH
  b2b_sf_con_rev AS (
    SELECT
      "b2b_sf_con_rev" AS revenue_type,
      FORMAT_DATE("%Y-%m", opp_close_date) AS date,
      line_item_high_level_product,
      line_item_type_of_business,
      ROUNd(SUM(line_item_total_price_conv), 2) AS revenue
    FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_con_rev_final_v2`
    where opp_close_date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
    GROUP BY 1, 2, 3, 4
  ),
  b2b_sf_non_sql_rev AS (
    SELECT
      "b2b_sf_non_sql_rev" AS revenue_type,
      FORMAT_DATE("%Y-%m", opp_close_date) AS date,
      line_item_high_level_product,
      line_item_type_of_business,
      ROUND(SUM(line_item_total_price_conv), 2) AS revenue
    FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_non_sql_rev_final_v2`
    where opp_close_date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
    GROUP BY 1, 2, 3, 4
  ),
  b2b_sf_sql_rev AS (
    SELECT
      "b2b_sf_sql_rev" AS revenue_type,
      FORMAT_DATE("%Y-%m", opp_close_date) AS date,
      line_item_high_level_product,
      line_item_type_of_business,
      ROUND(SUM(line_item_total_price_conv), 2) AS revenue
    FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_sql_rev_final_v2`
    where opp_close_date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
    GROUP BY 1, 2, 3, 4
  ),
  combined_data AS (
    SELECT * FROM b2b_sf_con_rev
    UNION ALL
    SELECT * FROM b2b_sf_sql_rev
    UNION ALL
    SELECT * FROM b2b_sf_non_sql_rev
  ),
  individual_kpis AS (
    SELECT
      date,
      -- --Marketing lead influenced revenue
      -- SUM(CASE WHEN line_item_type_of_business = "New" OR line_item_type_of_business IS NULL THEN revenue ELSE 0 END) AS mqlRevNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product != 'Journals'
            AND line_item_type_of_business
              IN ("Renewal", "Win Back", "Recurring")
            THEN revenue
          ELSE 0
          END) AS mqlRevRenewal,
      SUM(
        CASE WHEN line_item_type_of_business = "Upsell" THEN revenue ELSE 0 END)
        AS mqlRevUpsell,
      -- Individual B2B KPIs
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Journals'
            AND (
              line_item_type_of_business = "New"
              OR line_item_type_of_business IS NULL)
            THEN revenue
          ELSE 0
          END) AS mqlRevJrnlNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Journals'
            AND line_item_type_of_business = 'Upsell'
            THEN revenue
          ELSE 0
          END) AS mqlRevJrnlUpsell,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'eBooks'
            AND (
              line_item_type_of_business = "New"
              OR line_item_type_of_business IS NULL)
            THEN revenue
          ELSE 0
          END) AS mqlRevBksNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'eBooks'
            AND line_item_type_of_business = 'Upsell'
            THEN revenue
          ELSE 0
          END) AS mqlRevBksUpsell,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'eBooks'
            AND (
              line_item_type_of_business
              IN ("Renewal", "Win Back", "Recurring"))
            THEN revenue
          ELSE 0
          END) AS mqlRevBksRenewal,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Databases'
            AND (
              line_item_type_of_business = "New"
              OR line_item_type_of_business IS NULL)
            THEN revenue
          ELSE 0
          END) AS mqlRevDasNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Databases'
            AND line_item_type_of_business = 'Upsell'
            THEN revenue
          ELSE 0
          END) AS mqlRevDasUpsell,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Databases'
            AND (
              line_item_type_of_business
              IN ("Renewal", "Win Back", "Recurring"))
            THEN revenue
          ELSE 0
          END) AS mqlRevDasRenewal,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Other'
            AND (
              line_item_type_of_business = "New"
              OR line_item_type_of_business IS NULL)
            THEN revenue
          ELSE 0
          END) AS mqlRevOtherNew,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Other'
            AND line_item_type_of_business = 'Upsell'
            THEN revenue
          ELSE 0
          END) AS mqlRevOtherUpsell,
      SUM(
        CASE
          WHEN
            line_item_high_level_product = 'Other'
            AND (
              line_item_type_of_business
              IN ("Renewal", "Win Back", "Recurring"))
            THEN revenue
          ELSE 0
          END) AS mqlRevOtherRenewal
    FROM combined_data
    GROUP BY 1
  )
SELECT
  *,
  -- Combining individual B2B KPIs
  mqlRevJrnlNew + mqlRevJrnlUpsell AS mqlRevJrnl,
  mqlRevBksNew + mqlRevBksUpsell + mqlRevBksRenewal AS mqlRevBks,
  mqlRevDasNew + mqlRevDasUpsell + mqlRevDasRenewal AS mqlRevDas,
  mqlRevOtherNew + mqlRevOtherUpsell + mqlRevOtherRenewal AS mqlRevOther
FROM individual_kpis
ORDER BY 1 ASC

##JOIN tables into one
*   after individual KPIs have been grouped into the schema as in the table below perform an INNER JOIN to bring all KPIs into one table

Schema

*   date [string]
*   kpi1 [int]
*   kpi2 [int]<br>
...


In [ ]:
%%bigquery --project  snproject147bce46

create or replace view `snproject147bce46.2026_kpi_report.ecom_researcher_seo_mqlsJbd`

as

  SELECT *
  FROM `snproject147bce46.2026_kpi_report.ecom`

  FULL JOIN `snproject147bce46.2026_kpi_report.researcher`
  USING (date)

  FULL JOIN `snproject147bce46.2026_kpi_report.seo`
  USING (date)

  FULL JOIN `snproject147bce46.2026_kpi_report.books`
  USING (date)

  FULL JOIN `snproject147bce46.2026_kpi_report.mql_jbd`
  USING (date)

  FULL JOIN `snproject147bce46.2026_kpi_report.mqlRev_jbd`
  USING (date)

  FULL JOIN `snproject147bce46.2026_kpi_report.submissions`
  USING (date)

ORDER BY date ASC


Query is running:   0%|          |

""


##separate KPI values into acutal and py

*   extract month column

Schema

*   month [int]
*   kpi1_actual [int]
*   kpi1_py [int]
*   kpi2_actual [int]
*   kpi2_py [int]
<br>
...





In [ ]:
%%bigquery --project  snproject147bce46

CREATE OR REPLACE view `snproject147bce46.2026_kpi_report.actuals_py_gbq_kpis_combined`

as

SELECT
extract (month from cast (date as date format 'YYYY-MM')) as month,

IFNULL(SUM(ecomRevActMktSLNAT_py),0.0) as ecomRevActMktSLNAT_py,
IFNULL(SUM(ecomRevActMktSLNAT_actual),0.0) as ecomRevActMktSLNAT_actual,

IFNULL(SUM(ecomRevSeoSLNAT_py),0.0) as ecomRevSeoSLNAT_py,
IFNULL(SUM(ecomRevSeoSLNAT_actual),0.0) as ecomRevSeoSLNAT_actual,

IFNULL(SUM(ecomRevActMktSLbooksEbks_py),0.0) as ecomRevActMktSLbooksEbks_py,
IFNULL(SUM(ecomRevActMktSLbooksEbks_actual),0.0) as ecomRevActMktSLbooksEbks_actual,

IFNULL(SUM(ecomRevSeoSLbooksEbks_py),0.0) as ecomRevSeoSLbooksEbks_py,
IFNULL(SUM(ecomRevSeoSLbooksEbks_actual),0.0) as ecomRevSeoSLbooksEbks_actual,

IFNULL(SUM(ecomRevActMktSLppvchap_py),0.0) as ecomRevActMktSLppvchap_py,
IFNULL(SUM(ecomRevActMktSLppvchap_actual),0.0) as ecomRevActMktSLppvchap_actual,

IFNULL(SUM(ecomRevSeoSLppvchap_py),0.0) as ecomRevSeoSLppvchap_py,
IFNULL(SUM(ecomRevSeoSLppvchap_actual),0.0) as ecomRevSeoSLppvchap_actual,

IFNULL(SUM(ecomRevActMktSLppvart_py),0.0) as ecomRevActMktSLppvart_py,
IFNULL(SUM(ecomRevActMktSLppvart_actual),0.0) as ecomRevActMktSLppvart_actual,

IFNULL(SUM(ecomRevSeoSLppvart_py),0.0) as ecomRevSeoSLppvart_py,
IFNULL(SUM(ecomRevSeoSLppvart_actual),0.0) as ecomRevSeoSLppvart_actual,

IFNULL(SUM(ecomRevActMktSLsubs_py),0.0) as ecomRevActMktSLsubs_py,
IFNULL(SUM(ecomRevActMktSLsubs_actual),0.0) as ecomRevActMktSLsubs_actual,

IFNULL(SUM(ecomRevSeoSLsubs_py),0.0) as ecomRevSeoSLsubs_py,
IFNULL(SUM(ecomRevSeoSLsubs_actual),0.0) as ecomRevSeoSLsubs_actual,

IFNULL(SUM(ecomRevActMktB2BInst_py),0.0) as ecomRevActMktB2BInst_py,
IFNULL(SUM(ecomRevActMktB2BInst_actual),0.0) as ecomRevActMktB2BInst_actual,

IFNULL(SUM(ecomRevSeoB2BInst_py),0.0) as ecomRevSeoB2BInst_py,
IFNULL(SUM(ecomRevSeoB2BInst_actual),0.0) as ecomRevSeoB2BInst_actual,

IFNULL(SUM(ecomRevActMkt_py),0.0) as ecomRevActMkt_py,
IFNULL(SUM(ecomRevActMkt_actual),0.0) as ecomRevActMkt_actual,

IFNULL(SUM(ecomRevSeo_py),0.0) as ecomRevSeo_py,
IFNULL(SUM(ecomRevSeo_actual),0.0) as ecomRevSeo_actual,

IFNULL(SUM(ecomRev_py),0.0) as ecomRev_py,
IFNULL(SUM(ecomRev_actual),0.0) as ecomRev_actual,

IFNULL(SUM(submitClickMkt_py),0.0) as submitClickMkt_py,
IFNULL(SUM(submitClickMkt_actual),0.0) as submitClickMkt_actual,

IFNULL(SUM(submitClickMktOa_py),0.0) as submitClickMktOa_py,
IFNULL(SUM(submitClickMktOa_actual),0.0) as submitClickMktOa_actual,

IFNULL(SUM(elsMkt_py),0.0) as elsMkt_py,
IFNULL(SUM(elsMkt_actual),0.0) as elsMkt_actual,

IFNULL(SUM(elsMktOa_py),0.0) as elsMktOa_py,
IFNULL(SUM(elsMktOa_actual),0.0) as elsMktOa_actual,

IFNULL(SUM(JrnlEngMkt_py),0.0) as JrnlEngMkt_py,
IFNULL(SUM(JrnlEngMkt_actual),0.0) as JrnlEngMkt_actual,

IFNULL(SUM(JrnlEngMktOa_py),0.0) as JrnlEngMktOa_py,
IFNULL(SUM(JrnlEngMktOa_actual),0.0) as JrnlEngMktOa_actual,

IFNULL(SUM(seo_py),0.0) as seo_py,
IFNULL(SUM(seo_actual),0.0) as seo_actual,

IFNULL(SUM(AIseo_py),0.0) as AIseo_py,
IFNULL(SUM(AIseo_actual),0.0) as AIseo_actual,

IFNULL(SUM(Traditionalseo_py),0.0) as Traditionalseo_py,
IFNULL(SUM(Traditionalseo_actual),0.0) as Traditionalseo_actual,

IFNULL(SUM(bookLeads_py),0.0) as bookLeads_py,
IFNULL(SUM(bookLeads_actual),0.0) as bookLeads_actual,

IFNULL(SUM(bookLeadsAcc_py),0.0) as bookLeadsAcc_py,
IFNULL(SUM(bookLeadsAcc_actual),0.0) as bookLeadsAcc_actual,

IFNULL(SUM(bookCntctSgnMkt_py),0.0) as bookCntctSgnMkt_py,
IFNULL(SUM(bookCntctSgnMkt_actual),0.0) as bookCntctSgnMkt_actual,

IFNULL(SUM(salJrnl_py),0.0) as salJrnl_py,
IFNULL(SUM(salJrnl_actual),0.0) as salJrnl_actual,

IFNULL(SUM(salBks_py),0.0) as salBks_py,
IFNULL(SUM(salBks_actual),0.0) as salBks_actual,

IFNULL(SUM(salDas_py),0.0) as salDas_py,
IFNULL(SUM(salDas_actual),0.0) as salDas_actual,

IFNULL(SUM(mqlJrnl_py),0.0) as mqlJrnl_py,
IFNULL(SUM(mqlJrnl_actual),0.0) as mqlJrnl_actual,

IFNULL(SUM(mqlBks_py),0.0) as mqlBks_py,
IFNULL(SUM(mqlBks_actual),0.0) as mqlBks_actual,

IFNULL(SUM(mqlDas_py),0.0) as mqlDas_py,
IFNULL(SUM(mqlDas_actual),0.0) as mqlDas_actual,

IFNULL(SUM(mqlRevRenewal_py),0.0) as mqlRevRenewal_py,
IFNULL(SUM(mqlRevRenewal_actual),0.0) as mqlRevRenewal_actual,

IFNULL(SUM(mqlRevUpsell_py),0.0) as mqlRevUpsell_py,
IFNULL(SUM(mqlRevUpsell_actual),0.0) as mqlRevUpsell_actual,

IFNULL(SUM(mqlRevJrnlNew_py),0.0) as mqlRevJrnlNew_py,
IFNULL(SUM(mqlRevJrnlNew_actual),0.0) as mqlRevJrnlNew_actual,

IFNULL(SUM(mqlRevJrnlUpsell_py),0.0) as mqlRevJrnlUpsell_py,
IFNULL(SUM(mqlRevJrnlUpsell_actual),0.0) as mqlRevJrnlUpsell_actual,

IFNULL(SUM(mqlRevBksNew_py),0.0) as mqlRevBksNew_py,
IFNULL(SUM(mqlRevBksNew_actual),0.0) as mqlRevBksNew_actual,

IFNULL(SUM(mqlRevBksUpsell_py),0.0) as mqlRevBksUpsell_py,
IFNULL(SUM(mqlRevBksUpsell_actual),0.0) as mqlRevBksUpsell_actual,

IFNULL(SUM(mqlRevBksRenewal_py),0.0) as mqlRevBksRenewal_py,
IFNULL(SUM(mqlRevBksRenewal_actual),0.0) as mqlRevBksRenewal_actual,

IFNULL(SUM(mqlRevDasNew_py),0.0) as mqlRevDasNew_py,
IFNULL(SUM(mqlRevDasNew_actual),0.0) as mqlRevDasNew_actual,

IFNULL(SUM(mqlRevDasUpsell_py),0.0) as mqlRevDasUpsell_py,
IFNULL(SUM(mqlRevDasUpsell_actual),0.0) as mqlRevDasUpsell_actual,

IFNULL(SUM(mqlRevDasRenewal_py),0.0) as mqlRevDasRenewal_py,
IFNULL(SUM(mqlRevDasRenewal_actual),0.0) as mqlRevDasRenewal_actual,

IFNULL(SUM(mqlRevOtherNew_py),0.0) as mqlRevOtherNew_py,
IFNULL(SUM(mqlRevOtherNew_actual),0.0) as mqlRevOtherNew_actual,

IFNULL(SUM(mqlRevOtherUpsell_py),0.0) as mqlRevOtherUpsell_py,
IFNULL(SUM(mqlRevOtherUpsell_actual),0.0) as mqlRevOtherUpsell_actual,

IFNULL(SUM(mqlRevOtherRenewal_py),0.0) as mqlRevOtherRenewal_py,
IFNULL(SUM(mqlRevOtherRenewal_actual),0.0) as mqlRevOtherRenewal_actual,

IFNULL(SUM(mqlRevJrnl_py),0.0) as mqlRevJrnl_py,
IFNULL(SUM(mqlRevJrnl_actual),0.0) as mqlRevJrnl_actual,

IFNULL(SUM(mqlRevBks_py),0.0) as mqlRevBks_py,
IFNULL(SUM(mqlRevBks_actual),0.0) as mqlRevBks_actual,

IFNULL(SUM(mqlRevDas_py),0.0) as mqlRevDas_py,
IFNULL(SUM(mqlRevDas_actual),0.0) as mqlRevDas_actual,

IFNULL(SUM(mqlRevOther_py),0.0) as mqlRevOther_py,
IFNULL(SUM(mqlRevOther_actual),0.0) as mqlRevOther_actual,

IFNULL(SUM(submMktInfl1Plus_py),0.0) as submMktInfl1Plus_py,
IFNULL(SUM(submMktInfl1Plus_actual),0.0) as submMktInfl1Plus_actual,

IFNULL(SUM(submMktInflCol1Plus_py),0.0) as submMktInflCol1Plus_py,
IFNULL(SUM(submMktInflCol1Plus_actual),0.0) as submMktInflCol1Plus_actual,

IFNULL(SUM(submMktInfl3Plus_py),0.0) as submMktInfl3Plus_py,
IFNULL(SUM(submMktInfl3Plus_actual),0.0) as submMktInfl3Plus_actual,

IFNULL(SUM(submMktInfl3PlusOa_py),0.0) as submMktInfl3PlusOa_py,
IFNULL(SUM(submMktInfl3PlusOa_actual),0.0) as submMktInfl3PlusOa_actual,

IFNULL(SUM(submMktInfl3PlusOtr_py),0.0) as submMktInfl3PlusOtr_py,
IFNULL(SUM(submMktInfl3PlusOtr_actual),0.0) as submMktInfl3PlusOtr_actual,

IFNULL(SUM(submMktGen_py),0.0) as submMktGen_py,
IFNULL(SUM(submMktGen_actual),0.0) as submMktGen_actual,

IFNULL(SUM(submMktGenOa_py),0.0) as submMktGenOa_py,
IFNULL(SUM(submMktGenOa_actual),0.0) as submMktGenOa_actual,

IFNULL(SUM(powerAuthors_py),0.0) as powerAuthors_py,
IFNULL(SUM(powerAuthors_actual),0.0) as powerAuthors_actual

FROM
(
  SELECT
  date,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevActMktSLNAT end as ecomRevActMktSLNAT_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevSeoSLNAT end as ecomRevSeoSLNAT_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevActMktSLbooksEbks end as ecomRevActMktSLbooksEbks_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevSeoSLbooksEbks end as ecomRevSeoSLbooksEbks_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevActMktSLppvchap end as ecomRevActMktSLppvchap_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevSeoSLppvchap end as ecomRevSeoSLppvchap_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevActMktSLppvart end as ecomRevActMktSLppvart_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevSeoSLppvart end as ecomRevSeoSLppvart_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevActMktSLsubs end as ecomRevActMktSLsubs_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevSeoSLsubs end as ecomRevSeoSLsubs_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevActMktB2BInst end as ecomRevActMktB2BInst_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevSeoB2BInst end as ecomRevSeoB2BInst_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevActMkt end as ecomRevActMkt_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRevSeo end as ecomRevSeo_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then ecomRev end as ecomRev_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submitClickMkt end as submitClickMkt_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submitClickMktOa end as submitClickMktOa_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then elsMkt end as elsMkt_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then elsMktOa end as elsMktOa_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then JrnlEngMkt end as JrnlEngMkt_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then JrnlEngMktOa end as JrnlEngMktOa_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then seo end as seo_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then AIseo end as AIseo_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then Traditionalseo end as Traditionalseo_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then bookLeads end as bookLeads_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then bookLeadsAcc end as bookLeadsAcc_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then bookCntctSgnMkt end as bookCntctSgnMkt_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then salJrnl end as salJrnl_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then salBks end as salBks_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then salDas end as salDas_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlJrnl end as mqlJrnl_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlBks end as mqlBks_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlDas end as mqlDas_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevRenewal end as mqlRevRenewal_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevUpsell end as mqlRevUpsell_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevJrnlNew end as mqlRevJrnlNew_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevJrnlUpsell end as mqlRevJrnlUpsell_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevBksNew end as mqlRevBksNew_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevBksUpsell end as mqlRevBksUpsell_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevBksRenewal end as mqlRevBksRenewal_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevDasNew end as mqlRevDasNew_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevDasUpsell end as mqlRevDasUpsell_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevDasRenewal end as mqlRevDasRenewal_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevOtherNew end as mqlRevOtherNew_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevOtherUpsell end as mqlRevOtherUpsell_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevOtherRenewal end as mqlRevOtherRenewal_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevJrnl end as mqlRevJrnl_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevBks end as mqlRevBks_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevDas end as mqlRevDas_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then mqlRevOther end as mqlRevOther_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submMktInfl1Plus end as submMktInfl1Plus_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submMktInflCol1Plus end as submMktInflCol1Plus_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submMktInfl3Plus end as submMktInfl3Plus_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submMktInfl3PlusOa end as submMktInfl3PlusOa_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submMktInfl3PlusOtr end as submMktInfl3PlusOtr_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submMktGen end as submMktGen_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then submMktGenOa end as submMktGenOa_py,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2025 then powerAuthors end as powerAuthors_py,


  --actuals value
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevActMktSLNAT end as ecomRevActMktSLNAT_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevSeoSLNAT end as ecomRevSeoSLNAT_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevActMktSLbooksEbks end as ecomRevActMktSLbooksEbks_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevSeoSLbooksEbks end as ecomRevSeoSLbooksEbks_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevActMktSLppvchap end as ecomRevActMktSLppvchap_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevSeoSLppvchap end as ecomRevSeoSLppvchap_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevActMktSLppvart end as ecomRevActMktSLppvart_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevSeoSLppvart end as ecomRevSeoSLppvart_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevActMktSLsubs end as ecomRevActMktSLsubs_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevSeoSLsubs end as ecomRevSeoSLsubs_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevActMktB2BInst end as ecomRevActMktB2BInst_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevSeoB2BInst end as ecomRevSeoB2BInst_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevActMkt end as ecomRevActMkt_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRevSeo end as ecomRevSeo_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then ecomRev end as ecomRev_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submitClickMkt end as submitClickMkt_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submitClickMktOa end as submitClickMktOa_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then elsMkt end as elsMkt_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then elsMktOa end as elsMktOa_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then JrnlEngMkt end as JrnlEngMkt_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then JrnlEngMktOa end as JrnlEngMktOa_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then seo end as seo_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then AIseo end as AIseo_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then Traditionalseo end as Traditionalseo_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then bookLeads end as bookLeads_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then bookLeadsAcc end as bookLeadsAcc_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then bookCntctSgnMkt end as bookCntctSgnMkt_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then salJrnl end as salJrnl_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then salBks end as salBks_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then salDas end as salDas_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlJrnl end as mqlJrnl_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlBks end as mqlBks_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlDas end as mqlDas_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevRenewal end as mqlRevRenewal_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevUpsell end as mqlRevUpsell_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevJrnlNew end as mqlRevJrnlNew_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevJrnlUpsell end as mqlRevJrnlUpsell_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevBksNew end as mqlRevBksNew_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevBksUpsell end as mqlRevBksUpsell_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevBksRenewal end as mqlRevBksRenewal_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevDasNew end as mqlRevDasNew_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevDasUpsell end as mqlRevDasUpsell_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevDasRenewal end as mqlRevDasRenewal_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevOtherNew end as mqlRevOtherNew_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevOtherUpsell end as mqlRevOtherUpsell_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevOtherRenewal end as mqlRevOtherRenewal_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevJrnl end as mqlRevJrnl_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevBks end as mqlRevBks_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevDas end as mqlRevDas_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then mqlRevOther end as mqlRevOther_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submMktInfl1Plus end as submMktInfl1Plus_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submMktInflCol1Plus end as submMktInflCol1Plus_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submMktInfl3Plus end as submMktInfl3Plus_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submMktInfl3PlusOa end as submMktInfl3PlusOa_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submMktInfl3PlusOtr end as submMktInfl3PlusOtr_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submMktGen end as submMktGen_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then submMktGenOa end as submMktGenOa_actual,
    case when extract(Year from cast(date as date format 'YYYY-MM')) = 2026 then powerAuthors end as powerAuthors_actual,

  FROM `snproject147bce46.2026_kpi_report.ecom_researcher_seo_mqlsJbd`
)
group by month
order by month asc

Query is running:   0%|          |

""


##Join GBQ and Google Sheets actual and py tables

In [ ]:
%%bigquery --project  snproject147bce46
create or replace view `snproject147bce46.2026_kpi_report.all_actuals_py`

AS

SELECT *,

--mqlRevBksECom
cast(mqlRevBksNew_actual AS FLOAT64) + cast(mqlRevBksUpsell_actual AS FLOAT64) + cast(ecomRevActMktSLbooksEbks_actual AS FLOAT64) + cast(ecomRevSeoSLbooksEbks_actual AS FLOAT64) + cast(ecomRevActMktSLppvchap_actual AS FLOAT64) + cast(ecomRevSeoSLppvchap_actual AS FLOAT64) + cast(ifnull(ecomRevActMktB2BInst_actual,0) as float64) + cast(ifnull(ecomRevSeoB2BInst_actual,0) as float64)  AS mqlRevBksECom_actual,
cast(mqlRevBksNew_py AS FLOAT64) + cast(mqlRevBksUpsell_py AS FLOAT64) + cast(ecomRevActMktSLbooksEbks_py AS FLOAT64) + cast(ecomRevSeoSLbooksEbks_py AS FLOAT64) + cast(ecomRevActMktSLppvchap_py AS FLOAT64) + cast(ecomRevSeoSLppvchap_py AS FLOAT64) + CAST(ifnull(ecomRevActMktB2BInst_py,0) AS float64) + CAST(ifnull(ecomRevSeoB2BInst_py,0) AS float64) AS mqlRevBksECom_py,

--SALs
cast(cast(salJrnl_actual AS FLOAT64) + cast(salBks_actual AS FLOAT64) + cast(salDas_actual AS FLOAT64) + cast(salIs_actual AS FLOAT64) AS FLOAT64) as sal_actual,
cast(cast(salJrnl_py AS FLOAT64) + cast(salBks_py AS FLOAT64) + cast(salDas_py AS FLOAT64) + cast(salIs_py AS FLOAT64) AS FLOAT64) as sal_py,

--MQLs
CAST(cast(mqlJrnl_actual as FLOAT64) + cast(mqlBks_actual as FLOAT64)+ cast(mqlDas_actual as FLOAT64) + cast(mqlIs_actual AS FLOAT64) AS FLOAT64) as mql_actual,
CAST(cast(mqlJrnl_py as FLOAT64) + cast(mqlBks_py as FLOAT64)+ cast(mqlDas_py as FLOAT64) + cast(mqlIs_py AS FLOAT64) AS FLOAT64) as mql_py

FROM
(
  SELECT * except(Date) FROM `snproject147bce46.2026_kpi_report.actuals_py_gbq_kpis_combined` a
  FULL JOIN `snproject147bce46.2026_kpi_report.kpi_data_sheet` b
  on a.month = extract(month from b.date)
)
order by month

Query is running:   0%|          |

""


##PIVOT on KPI column

In [ ]:
%%bigquery --project  snproject147bce46

create or replace view `snproject147bce46.2026_kpi_report.pivot_actuals_py`
AS

SELECT
Parse_date("%Y-%m-%d", Concat("2026-", month, "-1")) AS date,
kpi,
Sum(actual) AS actual,
Sum(py) AS py
FROM
(
  SELECT
  month,
  kpi,
  CASE WHEN dimension = "py" THEN values END AS py,
  CASE WHEN dimension = "actual" THEN values END AS actual
  FROM
  (
    SELECT
    month,
    values,
    Split(test, "_")[offset(0)] as kpi,
    split(test, "_")[offset(1)] as dimension
    FROM
    (
      SELECT
      month,
      CAST(ecomRevActMktSLNAT_py AS FLOAT64) AS ecomRevActMktSLNAT_py,
      CAST(ecomRevActMktSLNAT_actual AS FLOAT64) AS ecomRevActMktSLNAT_actual,
      CAST(ecomRevSeoSLNAT_py AS FLOAT64) AS ecomRevSeoSLNAT_py,
      CAST(ecomRevSeoSLNAT_actual AS FLOAT64) AS ecomRevSeoSLNAT_actual,
      CAST(ecomRevActMktSLbooksEbks_py AS FLOAT64) AS ecomRevActMktSLbooksEbks_py,
      CAST(ecomRevActMktSLbooksEbks_actual AS FLOAT64) AS ecomRevActMktSLbooksEbks_actual,
      CAST(ecomRevSeoSLbooksEbks_py AS FLOAT64) AS ecomRevSeoSLbooksEbks_py,
      CAST(ecomRevSeoSLbooksEbks_actual AS FLOAT64) AS ecomRevSeoSLbooksEbks_actual,
      CAST(ecomRevActMktSLppvchap_py AS FLOAT64) AS ecomRevActMktSLppvchap_py,
      CAST(ecomRevActMktSLppvchap_actual AS FLOAT64) AS ecomRevActMktSLppvchap_actual,
      CAST(ecomRevSeoSLppvchap_py AS FLOAT64) AS ecomRevSeoSLppvchap_py,
      CAST(ecomRevSeoSLppvchap_actual AS FLOAT64) AS ecomRevSeoSLppvchap_actual,
      CAST(ecomRevActMktSLppvart_py AS FLOAT64) AS ecomRevActMktSLppvart_py,
      CAST(ecomRevActMktSLppvart_actual AS FLOAT64) AS ecomRevActMktSLppvart_actual,
      CAST(ecomRevSeoSLppvart_py AS FLOAT64) AS ecomRevSeoSLppvart_py,
      CAST(ecomRevSeoSLppvart_actual AS FLOAT64) AS ecomRevSeoSLppvart_actual,
      CAST(ecomRevActMktSLsubs_py AS FLOAT64) AS ecomRevActMktSLsubs_py,
      CAST(ecomRevActMktSLsubs_actual AS FLOAT64) AS ecomRevActMktSLsubs_actual,
      CAST(ecomRevSeoSLsubs_py AS FLOAT64) AS ecomRevSeoSLsubs_py,
      CAST(ecomRevSeoSLsubs_actual AS FLOAT64) AS ecomRevSeoSLsubs_actual,
      CAST(ecomRevActMktB2BInst_py AS FLOAT64) AS ecomRevActMktB2BInst_py,
      CAST(ecomRevActMktB2BInst_actual AS FLOAT64) AS ecomRevActMktB2BInst_actual,
      CAST(ecomRevSeoB2BInst_py AS FLOAT64) AS ecomRevSeoB2BInst_py,
      CAST(ecomRevSeoB2BInst_actual AS FLOAT64) AS ecomRevSeoB2BInst_actual,
      CAST(ecomRevActMkt_py AS FLOAT64) AS ecomRevActMkt_py,
      CAST(ecomRevActMkt_actual AS FLOAT64) AS ecomRevActMkt_actual,
      CAST(ecomRevSeo_py AS FLOAT64) AS ecomRevSeo_py,
      CAST(ecomRevSeo_actual AS FLOAT64) AS ecomRevSeo_actual,
      CAST(ecomRev_py AS FLOAT64) AS ecomRev_py,
      CAST(ecomRev_actual AS FLOAT64) AS ecomRev_actual,
      CAST(submitClickMkt_py AS FLOAT64) AS submitClickMkt_py,
      CAST(submitClickMkt_actual AS FLOAT64) AS submitClickMkt_actual,
      CAST(submitClickMktOa_py AS FLOAT64) AS submitClickMktOa_py,
      CAST(submitClickMktOa_actual AS FLOAT64) AS submitClickMktOa_actual,
      CAST(elsMkt_py AS FLOAT64) AS elsMkt_py,
      CAST(elsMkt_actual AS FLOAT64) AS elsMkt_actual,
      CAST(elsMktOa_py AS FLOAT64) AS elsMktOa_py,
      CAST(elsMktOa_actual AS FLOAT64) AS elsMktOa_actual,
      CAST(JrnlEngMkt_py AS FLOAT64) AS JrnlEngMkt_py,
      CAST(JrnlEngMkt_actual AS FLOAT64) AS JrnlEngMkt_actual,
      CAST(JrnlEngMktOa_py AS FLOAT64) AS JrnlEngMktOa_py,
      CAST(JrnlEngMktOa_actual AS FLOAT64) AS JrnlEngMktOa_actual,
      CAST(seo_py AS FLOAT64) AS seo_py,
      CAST(seo_actual AS FLOAT64) AS seo_actual,
      CAST(AIseo_py AS FLOAT64) AS AIseo_py,
      CAST(AIseo_actual AS FLOAT64) AS AIseo_actual,
      CAST(Traditionalseo_py AS FLOAT64) AS Traditionalseo_py,
      CAST(Traditionalseo_actual AS FLOAT64) AS Traditionalseo_actual,
      CAST(bookLeads_py AS FLOAT64) AS bookLeads_py,
      CAST(bookLeads_actual AS FLOAT64) AS bookLeads_actual,
      CAST(bookLeadsAcc_py AS FLOAT64) AS bookLeadsAcc_py,
      CAST(bookLeadsAcc_actual AS FLOAT64) AS bookLeadsAcc_actual,
      CAST(bookCntctSgnMkt_py AS FLOAT64) AS bookCntctSgnMkt_py,
      CAST(bookCntctSgnMkt_actual AS FLOAT64) AS bookCntctSgnMkt_actual,
      CAST(salJrnl_py AS FLOAT64) AS salJrnl_py,
      CAST(salJrnl_actual AS FLOAT64) AS salJrnl_actual,
      CAST(salBks_py AS FLOAT64) AS salBks_py,
      CAST(salBks_actual AS FLOAT64) AS salBks_actual,
      CAST(salDas_py AS FLOAT64) AS salDas_py,
      CAST(salDas_actual AS FLOAT64) AS salDas_actual,
      CAST(mqlJrnl_py AS FLOAT64) AS mqlJrnl_py,
      CAST(mqlJrnl_actual AS FLOAT64) AS mqlJrnl_actual,
      CAST(mqlBks_py AS FLOAT64) AS mqlBks_py,
      CAST(mqlBks_actual AS FLOAT64) AS mqlBks_actual,
      CAST(mqlDas_py AS FLOAT64) AS mqlDas_py,
      CAST(mqlDas_actual AS FLOAT64) AS mqlDas_actual,
      CAST(mqlRevRenewal_py AS FLOAT64) AS mqlRevRenewal_py,
      CAST(mqlRevRenewal_actual AS FLOAT64) AS mqlRevRenewal_actual,
      CAST(mqlRevUpsell_py AS FLOAT64) AS mqlRevUpsell_py,
      CAST(mqlRevUpsell_actual AS FLOAT64) AS mqlRevUpsell_actual,
      CAST(mqlRevJrnlNew_py AS FLOAT64) AS mqlRevJrnlNew_py,
      CAST(mqlRevJrnlNew_actual AS FLOAT64) AS mqlRevJrnlNew_actual,
      CAST(mqlRevJrnlUpsell_py AS FLOAT64) AS mqlRevJrnlUpsell_py,
      CAST(mqlRevJrnlUpsell_actual AS FLOAT64) AS mqlRevJrnlUpsell_actual,
      CAST(mqlRevBksNew_py AS FLOAT64) AS mqlRevBksNew_py,
      CAST(mqlRevBksNew_actual AS FLOAT64) AS mqlRevBksNew_actual,
      CAST(mqlRevBksUpsell_py AS FLOAT64) AS mqlRevBksUpsell_py,
      CAST(mqlRevBksUpsell_actual AS FLOAT64) AS mqlRevBksUpsell_actual,
      CAST(mqlRevBksRenewal_py AS FLOAT64) AS mqlRevBksRenewal_py,
      CAST(mqlRevBksRenewal_actual AS FLOAT64) AS mqlRevBksRenewal_actual,
      CAST(mqlRevDasNew_py AS FLOAT64) AS mqlRevDasNew_py,
      CAST(mqlRevDasNew_actual AS FLOAT64) AS mqlRevDasNew_actual,
      CAST(mqlRevDasUpsell_py AS FLOAT64) AS mqlRevDasUpsell_py,
      CAST(mqlRevDasUpsell_actual AS FLOAT64) AS mqlRevDasUpsell_actual,
      CAST(mqlRevDasRenewal_py AS FLOAT64) AS mqlRevDasRenewal_py,
      CAST(mqlRevDasRenewal_actual AS FLOAT64) AS mqlRevDasRenewal_actual,
      CAST(mqlRevOtherNew_py AS FLOAT64) AS mqlRevOtherNew_py,
      CAST(mqlRevOtherNew_actual AS FLOAT64) AS mqlRevOtherNew_actual,
      CAST(mqlRevOtherUpsell_py AS FLOAT64) AS mqlRevOtherUpsell_py,
      CAST(mqlRevOtherUpsell_actual AS FLOAT64) AS mqlRevOtherUpsell_actual,
      CAST(mqlRevOtherRenewal_py AS FLOAT64) AS mqlRevOtherRenewal_py,
      CAST(mqlRevOtherRenewal_actual AS FLOAT64) AS mqlRevOtherRenewal_actual,
      CAST(mqlRevJrnl_py AS FLOAT64) AS mqlRevJrnl_py,
      CAST(mqlRevJrnl_actual AS FLOAT64) AS mqlRevJrnl_actual,
      CAST(mqlRevBks_py AS FLOAT64) AS mqlRevBks_py,
      CAST(mqlRevBks_actual AS FLOAT64) AS mqlRevBks_actual,
      CAST(mqlRevDas_py AS FLOAT64) AS mqlRevDas_py,
      CAST(mqlRevDas_actual AS FLOAT64) AS mqlRevDas_actual,
      CAST(mqlRevOther_py AS FLOAT64) AS mqlRevOther_py,
      CAST(mqlRevOther_actual AS FLOAT64) AS mqlRevOther_actual,
      CAST(submMktInfl1Plus_py AS FLOAT64) AS submMktInfl1Plus_py,
      CAST(submMktInfl1Plus_actual AS FLOAT64) AS submMktInfl1Plus_actual,
      CAST(submMktInflCol1Plus_py AS FLOAT64) AS submMktInflCol1Plus_py,
      CAST(submMktInflCol1Plus_actual AS FLOAT64) AS submMktInflCol1Plus_actual,
      CAST(submMktInfl3Plus_py AS FLOAT64) AS submMktInfl3Plus_py,
      CAST(submMktInfl3Plus_actual AS FLOAT64) AS submMktInfl3Plus_actual,
      CAST(submMktInfl3PlusOa_py AS FLOAT64) AS submMktInfl3PlusOa_py,
      CAST(submMktInfl3PlusOa_actual AS FLOAT64) AS submMktInfl3PlusOa_actual,
      CAST(submMktInfl3PlusOtr_py AS FLOAT64) AS submMktInfl3PlusOtr_py,
      CAST(submMktInfl3PlusOtr_actual AS FLOAT64) AS submMktInfl3PlusOtr_actual,
      CAST(submMktGen_py AS FLOAT64) AS submMktGen_py,
      CAST(submMktGen_actual AS FLOAT64) AS submMktGen_actual,
      CAST(submMktGenOa_py AS FLOAT64) AS submMktGenOa_py,
      CAST(submMktGenOa_actual AS FLOAT64) AS submMktGenOa_actual,
      CAST(powerAuthors_py AS FLOAT64) AS powerAuthors_py,
      CAST(powerAuthors_actual AS FLOAT64) AS powerAuthors_actual,
      CAST(mktIndirCost_actual AS FLOAT64) AS mktIndirCost_actual,
      CAST(mktIndirCost_py AS FLOAT64) AS mktIndirCost_py,
      CAST(mktIndirCostPers_actual AS FLOAT64) AS mktIndirCostPers_actual,
      CAST(mktIndirCostPers_py AS FLOAT64) AS mktIndirCostPers_py,
      CAST(mktIndirCostMktExp_actual AS FLOAT64) AS mktIndirCostMktExp_actual,
      CAST(mktIndirCostMktExp_py AS FLOAT64) AS mktIndirCostMktExp_py,
      CAST(fte_actual AS FLOAT64) AS fte_actual,
      CAST(fte_py AS FLOAT64) AS fte_py,
      CAST(b2cCntct_actual AS FLOAT64) AS b2cCntct_actual,
      CAST(b2cCntct_py AS FLOAT64) AS b2cCntct_py,
      CAST(b2cCntctOptin_actual AS FLOAT64) AS b2cCntctOptin_actual,
      CAST(b2cCntctOptin_py AS FLOAT64) AS b2cCntctOptin_py,
      CAST(b2cAuthorsJrnl_actual AS FLOAT64) AS b2cAuthorsJrnl_actual,
      CAST(b2cAuthorsJrnl_py AS FLOAT64) AS b2cAuthorsJrnl_py,
      CAST(b2cAuthorsBks_actual AS FLOAT64) AS b2cAuthorsBks_actual,
      CAST(b2cAuthorsBks_py AS FLOAT64) AS b2cAuthorsBks_py,
      CAST(b2bCntct_actual AS FLOAT64) AS b2bCntct_actual,
      CAST(b2bCntct_py AS FLOAT64) AS b2bCntct_py,
      CAST(mqlRevIs_actual AS FLOAT64) AS mqlRevIs_actual,
      CAST(mqlRevIs_py AS FLOAT64) AS mqlRevIs_py,
      CAST(salIs_actual AS FLOAT64) AS salIs_actual,
      CAST(salIs_py AS FLOAT64) AS salIs_py,
      CAST(mqlIs_actual AS FLOAT64) AS mqlIs_actual,
      CAST(mqlIs_py AS FLOAT64) AS mqlIs_py,
      CAST(ecomRevNatCarBDJ_actual AS FLOAT64) AS ecomRevNatCarBDJ_actual,
      CAST(ecomRevNatCarBDJ_py AS FLOAT64) AS ecomRevNatCarBDJ_py,
      CAST(sourcePageviews_actual AS FLOAT64) AS sourcePageviews_actual,
      CAST(sourcePageviews_py AS FLOAT64) AS sourcePageviews_py,
      CAST(sourcePageviewsMktOptin_actual AS FLOAT64) AS sourcePageviewsMktOptin_actual,
      CAST(sourcePageviewsMktOptin_py AS FLOAT64) AS sourcePageviewsMktOptin_py,
      CAST(oaAgreements_actual AS FLOAT64) AS oaAgreements_actual,
      CAST(oaAgreements_py AS FLOAT64) AS oaAgreements_py,
      CAST(oaAgreementsEligAuthorSubm_actual AS FLOAT64) AS oaAgreementsEligAuthorSubm_actual,
      CAST(oaAgreementsEligAuthorSubm_py AS FLOAT64) AS oaAgreementsEligAuthorSubm_py,
      CAST(ecomRevActMktB2CsubsLTV_actual AS FLOAT64) AS ecomRevActMktB2CsubsLTV_actual,
      CAST(ecomRevActMktB2CsubsLTV_py AS FLOAT64) AS ecomRevActMktB2CsubsLTV_py,
      CAST(productAWA_actual AS FLOAT64) AS productAWA_actual,
      CAST(productAWA_py AS FLOAT64) AS productAWA_py,
      CAST(productAWAeTOCnat_actual AS FLOAT64) AS productAWAeTOCnat_actual,
      CAST(productAWAeTOCnat_py AS FLOAT64) AS productAWAeTOCnat_py,
      CAST(productAWAeTOCspr_actual AS FLOAT64) AS productAWAeTOCspr_actual,
      CAST(productAWAeTOCspr_py AS FLOAT64) AS productAWAeTOCspr_py,
      CAST(productAWASourceSubs_actual AS FLOAT64) AS productAWASourceSubs_actual,
      CAST(productAWASourceSubs_py AS FLOAT64) AS productAWASourceSubs_py,
      CAST(productAWABriefingSubs_actual AS FLOAT64) AS productAWABriefingSubs_actual,
      CAST(productAWABriefingSubs_py AS FLOAT64) AS productAWABriefingSubs_py,
      CAST(mqlRevBksECom_actual AS FLOAT64) AS mqlRevBksECom_actual,
      CAST(mqlRevBksECom_py AS FLOAT64) AS mqlRevBksECom_py,
      CAST(sal_actual AS FLOAT64) AS sal_actual,
      CAST(sal_py AS FLOAT64) AS sal_py,
      CAST(mql_actual AS FLOAT64) AS mql_actual,
      CAST(mql_py AS FLOAT64) AS mql_py,
      CAST(ImprintHubTraffic_actual AS FLOAT64) AS ImprintHubTraffic_actual,
      CAST(ImprintHubTraffic_py AS FLOAT64) AS ImprintHubTraffic_py

      FROM `snproject147bce46.2026_kpi_report.all_actuals_py`
    )
  UNPIVOT
      (
        values for test IN
        (
      ecomRevActMktSLNAT_py,
      ecomRevActMktSLNAT_actual,
      ecomRevSeoSLNAT_py,
      ecomRevSeoSLNAT_actual,
      ecomRevActMktSLbooksEbks_py,
      ecomRevActMktSLbooksEbks_actual,
      ecomRevSeoSLbooksEbks_py,
      ecomRevSeoSLbooksEbks_actual,
      ecomRevActMktSLppvchap_py,
      ecomRevActMktSLppvchap_actual,
      ecomRevSeoSLppvchap_py,
      ecomRevSeoSLppvchap_actual,
      ecomRevActMktSLppvart_py,
      ecomRevActMktSLppvart_actual,
      ecomRevSeoSLppvart_py,
      ecomRevSeoSLppvart_actual,
      ecomRevActMktSLsubs_py,
      ecomRevActMktSLsubs_actual,
      ecomRevSeoSLsubs_py,
      ecomRevSeoSLsubs_actual,
      ecomRevActMktB2BInst_py,
      ecomRevActMktB2BInst_actual,
      ecomRevSeoB2BInst_py,
      ecomRevSeoB2BInst_actual,
      ecomRevActMkt_py,
      ecomRevActMkt_actual,
      ecomRevSeo_py,
      ecomRevSeo_actual,
      ecomRev_py,
      ecomRev_actual,
      submitClickMkt_py,
      submitClickMkt_actual,
      submitClickMktOa_py,
      submitClickMktOa_actual,
      elsMkt_py,
      elsMkt_actual,
      elsMktOa_py,
      elsMktOa_actual,
      JrnlEngMkt_py,
      JrnlEngMkt_actual,
      JrnlEngMktOa_py,
      JrnlEngMktOa_actual,
      seo_py,
      seo_actual,
      AIseo_py,
      AIseo_actual,
      Traditionalseo_py,
      Traditionalseo_actual,
      bookLeads_py,
      bookLeads_actual,
      bookLeadsAcc_py,
      bookLeadsAcc_actual,
      bookCntctSgnMkt_py,
      bookCntctSgnMkt_actual,
      salJrnl_py,
      salJrnl_actual,
      salBks_py,
      salBks_actual,
      salDas_py,
      salDas_actual,
      mqlJrnl_py,
      mqlJrnl_actual,
      mqlBks_py,
      mqlBks_actual,
      mqlDas_py,
      mqlDas_actual,
      mqlRevRenewal_py,
      mqlRevRenewal_actual,
      mqlRevUpsell_py,
      mqlRevUpsell_actual,
      mqlRevJrnlNew_py,
      mqlRevJrnlNew_actual,
      mqlRevJrnlUpsell_py,
      mqlRevJrnlUpsell_actual,
      mqlRevBksNew_py,
      mqlRevBksNew_actual,
      mqlRevBksUpsell_py,
      mqlRevBksUpsell_actual,
      mqlRevBksRenewal_py,
      mqlRevBksRenewal_actual,
      mqlRevDasNew_py,
      mqlRevDasNew_actual,
      mqlRevDasUpsell_py,
      mqlRevDasUpsell_actual,
      mqlRevDasRenewal_py,
      mqlRevDasRenewal_actual,
      mqlRevOtherNew_py,
      mqlRevOtherNew_actual,
      mqlRevOtherUpsell_py,
      mqlRevOtherUpsell_actual,
      mqlRevOtherRenewal_py,
      mqlRevOtherRenewal_actual,
      mqlRevJrnl_py,
      mqlRevJrnl_actual,
      mqlRevBks_py,
      mqlRevBks_actual,
      mqlRevDas_py,
      mqlRevDas_actual,
      mqlRevOther_py,
      mqlRevOther_actual,
      submMktInfl1Plus_py,
      submMktInfl1Plus_actual,
      submMktInflCol1Plus_py,
      submMktInflCol1Plus_actual,
      submMktInfl3Plus_py,
      submMktInfl3Plus_actual,
      submMktInfl3PlusOa_py,
      submMktInfl3PlusOa_actual,
      submMktInfl3PlusOtr_py,
      submMktInfl3PlusOtr_actual,
      submMktGen_py,
      submMktGen_actual,
      submMktGenOa_py,
      submMktGenOa_actual,
      powerAuthors_py,
      powerAuthors_actual,
      mktIndirCost_actual,
      mktIndirCost_py,
      mktIndirCostPers_actual,
      mktIndirCostPers_py,
      mktIndirCostMktExp_actual,
      mktIndirCostMktExp_py,
      fte_actual,
      fte_py,
      b2cCntct_actual,
      b2cCntct_py,
      b2cCntctOptin_actual,
      b2cCntctOptin_py,
      b2cAuthorsJrnl_actual,
      b2cAuthorsJrnl_py,
      b2cAuthorsBks_actual,
      b2cAuthorsBks_py,
      b2bCntct_actual,
      b2bCntct_py,
      mqlRevIs_actual,
      mqlRevIs_py,
      salIs_actual,
      salIs_py,
      mqlIs_actual,
      mqlIs_py,
      ecomRevNatCarBDJ_actual,
      ecomRevNatCarBDJ_py,
      sourcePageviews_actual,
      sourcePageviews_py,
      sourcePageviewsMktOptin_actual,
      sourcePageviewsMktOptin_py,
      oaAgreements_actual,
      oaAgreements_py,
      oaAgreementsEligAuthorSubm_actual,
      oaAgreementsEligAuthorSubm_py,
      ecomRevActMktB2CsubsLTV_actual,
      ecomRevActMktB2CsubsLTV_py,
      productAWA_actual,
      productAWA_py,
      productAWAeTOCnat_actual,
      productAWAeTOCnat_py,
      productAWAeTOCspr_actual,
      productAWAeTOCspr_py,
      productAWASourceSubs_actual,
      productAWASourceSubs_py,
      productAWABriefingSubs_actual,
      productAWABriefingSubs_py,
      mqlRevBksECom_actual,
      mqlRevBksECom_py,
      sal_actual,
      sal_py,
      mql_actual,
      mql_py,
      ImprintHubTraffic_actual,
      ImprintHubTraffic_py
      )))
)
GROUP BY month, kpi
ORDER BY month ASC

Query is running:   0%|          |

""


## add order number

In [ ]:
%%bigquery --project  snproject147bce46

create or replace view `snproject147bce46.2026_kpi_report.pivot_actuals_py_order`

AS

SELECT
date,
kpi,

CASE
    WHEN kpi = 'mqlRevBksECom' THEN 1
    WHEN kpi = 'mktIndirCost' THEN 2
    WHEN kpi = 'mktIndirCostPers' THEN 3
    WHEN kpi = 'mktIndirCostMktExp' THEN 4
    WHEN kpi = 'submMktInfl3Plus' THEN 5
    WHEN kpi = 'bookCntctSgnMkt' THEN 6
    WHEN kpi = 'fte' THEN 7
    WHEN kpi = 'b2cCntct' THEN 8
    WHEN kpi = 'b2cCntctOptin' THEN 9
    WHEN kpi = 'b2cAuthorsJrnl' THEN 10
    WHEN kpi = 'b2cAuthorsBks' THEN 11
    WHEN kpi = 'b2bCntct' THEN 12
    WHEN kpi = 'mqlRevJrnl' THEN 13
    WHEN kpi = 'mqlRevJrnlNew' THEN 14
    WHEN kpi = 'mqlRevJrnlUpsell' THEN 15
    WHEN kpi = 'mqlRevBks' THEN 16
    WHEN kpi = 'mqlRevBksNew' THEN 17
    WHEN kpi = 'mqlRevBksUpsell' THEN 18
    WHEN kpi = 'mqlRevBksRenewal' THEN 19
    WHEN kpi = 'mqlRevDas' THEN 20
    WHEN kpi = 'mqlRevDasNew' THEN 21
    WHEN kpi = 'mqlRevDasUpsell' THEN 22
    WHEN kpi = 'mqlRevDasRenewal' THEN 23
    WHEN kpi = 'mqlRevIs' THEN 24
    WHEN kpi = 'sal' THEN 25
    WHEN kpi = 'salJrnl' THEN 26
    WHEN kpi = 'salBks' THEN 27
    WHEN kpi = 'salDas' THEN 28
    WHEN kpi = 'salIs' THEN 29
    WHEN kpi = 'mql' THEN 30
    WHEN kpi = 'ecomRev' THEN 31
    WHEN kpi = 'ecomRevActMkt' THEN 32
    WHEN kpi = 'ecomRevActMktSLNAT' THEN 33
    WHEN kpi = 'ecomRevActMktB2BInst' THEN 34
    WHEN kpi = 'ecomRevSeo' THEN 35
    WHEN kpi = 'ecomRevSeoSLNAT' THEN 36
    WHEN kpi = 'ecomRevSeoSLbooksEbks' THEN 37
    WHEN kpi = 'ecomRevSeoSLppvchap' THEN 38
    WHEN kpi = 'ecomRevSeoSLppvart' THEN 39
    WHEN kpi = 'ecomRevSeoSLsubs' THEN 40
    WHEN kpi = 'ecomRevSeoB2BInst' THEN 41
    WHEN kpi = 'ecomRevActMktB2CsubsLTV' THEN 42
    WHEN kpi = 'ecomRevNatCarBDJ' THEN 43
    WHEN kpi = 'seo' THEN 44
    WHEN kpi = 'Traditionalseo' THEN 45
    WHEN kpi = 'AIseo' THEN 46
    WHEN kpi = 'sourcePageviews' THEN 47
    WHEN kpi = 'sourcePageviewsMktOptin' THEN 48
    WHEN kpi = 'productAWA' THEN 49
    WHEN kpi = 'productAWAeTOCnat' THEN 50
    WHEN kpi = 'productAWAeTOCspr' THEN 51
    WHEN kpi = 'productAWASourceSubs' THEN 52
    WHEN kpi = 'productAWABriefingSubs' THEN 53
    WHEN kpi = 'JrnlEngMkt' THEN 54
    WHEN kpi = 'JrnlEngMktOa' THEN 55
    WHEN kpi = 'elsMkt' THEN 56
    WHEN kpi = 'elsMktOa' THEN 57
    WHEN kpi = 'submitClickMkt' THEN 58
    WHEN kpi = 'submitClickMktOa' THEN 59
    WHEN kpi = 'submMktInflCol1Plus' THEN 60
    WHEN kpi = 'submMktInfl3PlusOa' THEN 61
    WHEN kpi = 'submMktInfl3PlusOtr' THEN 62
    WHEN kpi = 'powerAuthors' THEN 63
    WHEN kpi = 'submMktGen' THEN 64
    WHEN kpi = 'submMktGenOa' THEN 65
    WHEN kpi = 'oaAgreements' THEN 66
    WHEN kpi = 'oaAgreementsEligAuthorSubm' THEN 67
    WHEN kpi = 'bookLeadsAcc' THEN 68
    WHEN kpi = 'bookLeads' THEN 69
    WHEN kpi = 'ImprintHubTraffic' THEN 70
    WHEN kpi = 'ecomRevActMktSLbooksEbks' THEN 71
    WHEN kpi = 'ecomRevActMktSLppvchap' THEN 72
    WHEN kpi = 'ecomRevActMktSLppvart' THEN 73
    WHEN kpi = 'ecomRevActMktSLsubs' THEN 74
    WHEN kpi = 'mqlJrnl' THEN 75
    WHEN kpi = 'mqlBks' THEN 76
    WHEN kpi = 'mqlDas' THEN 77
    WHEN kpi = 'mqlIs' THEN 78
    WHEN kpi = 'mqlRevRenewal' THEN 79
    WHEN kpi = 'mqlRevUpsell' THEN 80
    WHEN kpi = 'mqlRevOtherNew' THEN 81
    WHEN kpi = 'mqlRevOtherUpsell' THEN 82
    WHEN kpi = 'mqlRevOtherRenewal' THEN 83
    WHEN kpi = 'mqlRevOther' THEN 84
    WHEN kpi = 'submMktInfl1Plus' THEN 85

END as ordernumber,
actual,
py

FROM `snproject147bce46.2026_kpi_report.pivot_actuals_py`

order by date asc, ordernumber asc

Query is running:   0%|          |

""


##budget PIVOT

In [ ]:
%%bigquery --project  snproject147bce46

create or replace view `snproject147bce46.2026_kpi_report.pivot_budget`

as

SELECT
Parse_date("%Y-%m-%d", Concat("2026-", month, "-1")) AS date,
kpi,
Sum(budget) AS budget

FROM
(
  SELECT
  month,
  kpi,
  CASE WHEN dimension = "budget" THEN values END AS budget

  FROM
  (
    SELECT
    month,
    values,
    Split(test, "_")[offset(0)] as kpi,
    split(test, "_")[offset(1)] as dimension

    FROM

    (
      SELECT
      extract (month from date) as month,
      CAST(mqlRevBksECom_budget AS FLOAT64) AS mqlRevBksECom_budget,
      CAST(mktIndirCost_budget AS FLOAT64) AS mktIndirCost_budget,
      CAST(mktIndirCostPers_budget AS FLOAT64) AS mktIndirCostPers_budget,
      CAST(mktIndirCostMktExp_budget AS FLOAT64) AS mktIndirCostMktExp_budget,
      CAST(submMktInfl3Plus_budget AS FLOAT64) AS submMktInfl3Plus_budget,
      CAST(bookCntctSgnMkt_budget AS FLOAT64) AS bookCntctSgnMkt_budget,
      CAST(fte_budget AS FLOAT64) AS fte_budget,
      CAST(b2cCntct_budget AS FLOAT64) AS b2cCntct_budget,
      CAST(b2cCntctOptin_budget AS FLOAT64) AS b2cCntctOptin_budget,
      CAST(b2cAuthorsJrnl_budget AS FLOAT64) AS b2cAuthorsJrnl_budget,
      CAST(b2cAuthorsBks_budget AS FLOAT64) AS b2cAuthorsBks_budget,
      CAST(b2bCntct_budget AS FLOAT64) AS b2bCntct_budget,
      CAST(mqlRevJrnl_budget AS FLOAT64) AS mqlRevJrnl_budget,
      CAST(mqlRevJrnlNew_budget AS FLOAT64) AS mqlRevJrnlNew_budget,
      CAST(mqlRevJrnlUpsell_budget AS FLOAT64) AS mqlRevJrnlUpsell_budget,
      CAST(mqlRevBks_budget AS FLOAT64) AS mqlRevBks_budget,
      CAST(mqlRevBksNew_budget AS FLOAT64) AS mqlRevBksNew_budget,
      CAST(mqlRevBksUpsell_budget AS FLOAT64) AS mqlRevBksUpsell_budget,
      CAST(mqlRevBksRenewal_budget AS FLOAT64) AS mqlRevBksRenewal_budget,
      CAST(mqlRevDas_budget AS FLOAT64) AS mqlRevDas_budget,
      CAST(mqlRevDasNew_budget AS FLOAT64) AS mqlRevDasNew_budget,
      CAST(mqlRevDasUpsell_budget AS FLOAT64) AS mqlRevDasUpsell_budget,
      CAST(mqlRevDasRenewal_budget AS FLOAT64) AS mqlRevDasRenewal_budget,
      CAST(mqlRevIs_budget AS FLOAT64) AS mqlRevIs_budget,
      CAST(sal_budget AS FLOAT64) AS sal_budget,
      CAST(salJrnl_budget AS FLOAT64) AS salJrnl_budget,
      CAST(salBks_budget AS FLOAT64) AS salBks_budget,
      CAST(salDas_budget AS FLOAT64) AS salDas_budget,
      CAST(salIs_budget AS FLOAT64) AS salIs_budget,
      CAST(mql_budget AS FLOAT64) AS mql_budget,
      CAST(ecomRev_budget AS FLOAT64) AS ecomRev_budget,
      CAST(ecomRevActMkt_budget AS FLOAT64) AS ecomRevActMkt_budget,
      CAST(ecomRevActMktSLNAT_budget AS FLOAT64) AS ecomRevActMktSLNAT_budget,
      CAST(ecomRevActMktB2BInst_budget AS FLOAT64) AS ecomRevActMktB2BInst_budget,
      CAST(ecomRevSeo_budget AS FLOAT64) AS ecomRevSeo_budget,
      CAST(ecomRevSeoSLNAT_budget AS FLOAT64) AS ecomRevSeoSLNAT_budget,
      CAST(ecomRevSeoSLbooksEbks_budget AS FLOAT64) AS ecomRevSeoSLbooksEbks_budget,
      CAST(ecomRevSeoSLppvchap_budget AS FLOAT64) AS ecomRevSeoSLppvchap_budget,
      CAST(ecomRevSeoSLppvart_budget AS FLOAT64) AS ecomRevSeoSLppvart_budget,
      CAST(ecomRevSeoSLsubs_budget AS FLOAT64) AS ecomRevSeoSLsubs_budget,
      CAST(ecomRevSeoB2BInst_budget AS FLOAT64) AS ecomRevSeoB2BInst_budget,
      CAST(ecomRevActMktB2CsubsLTV_budget AS FLOAT64) AS ecomRevActMktB2CsubsLTV_budget,
      CAST(ecomRevNatCarBDJ_budget AS FLOAT64) AS ecomRevNatCarBDJ_budget,
      CAST(seo_budget AS FLOAT64) AS seo_budget,
      CAST(Traditionalseo_budget AS FLOAT64) AS Traditionalseo_budget,
      CAST(AIseo_budget AS FLOAT64) AS AIseo_budget,
      CAST(sourcePageviews_budget AS FLOAT64) AS sourcePageviews_budget,
      CAST(sourcePageviewsMktOptin_budget AS FLOAT64) AS sourcePageviewsMktOptin_budget,
      CAST(productAWA_budget AS FLOAT64) AS productAWA_budget,
      CAST(productAWAeTOCnat_budget AS FLOAT64) AS productAWAeTOCnat_budget,
      CAST(productAWAeTOCspr_budget AS FLOAT64) AS productAWAeTOCspr_budget,
      CAST(productAWASourceSubs_budget AS FLOAT64) AS productAWASourceSubs_budget,
      CAST(productAWABriefingSubs_budget AS FLOAT64) AS productAWABriefingSubs_budget,
      CAST(JrnlEngMkt_budget AS FLOAT64) AS JrnlEngMkt_budget,
      CAST(JrnlEngMktOa_budget AS FLOAT64) AS JrnlEngMktOa_budget,
      CAST(elsMkt_budget AS FLOAT64) AS elsMkt_budget,
      CAST(elsMktOa_budget AS FLOAT64) AS elsMktOa_budget,
      CAST(submitClickMkt_budget AS FLOAT64) AS submitClickMkt_budget,
      CAST(submitClickMktOa_budget AS FLOAT64) AS submitClickMktOa_budget,
      CAST(submMktInflCol1Plus_budget AS FLOAT64) AS submMktInflCol1Plus_budget,
      CAST(submMktInfl3PlusOa_budget AS FLOAT64) AS submMktInfl3PlusOa_budget,
      CAST(submMktInfl3PlusOtr_budget AS FLOAT64) AS submMktInfl3PlusOtr_budget,
      CAST(powerAuthors_budget AS FLOAT64) AS powerAuthors_budget,
      CAST(submMktGen_budget AS FLOAT64) AS submMktGen_budget,
      CAST(submMktGenOa_budget AS FLOAT64) AS submMktGenOa_budget,
      CAST(oaAgreements_budget AS FLOAT64) AS oaAgreements_budget,
      CAST(oaAgreementsEligAuthorSubm_budget AS FLOAT64) AS oaAgreementsEligAuthorSubm_budget,
      CAST(bookLeadsAcc_budget AS FLOAT64) AS bookLeadsAcc_budget,
      CAST(bookLeads_budget AS FLOAT64) AS bookLeads_budget,
      CAST(ecomRevActMktSLbooksEbks_budget AS FLOAT64) AS ecomRevActMktSLbooksEbks_budget,
      CAST(ecomRevActMktSLppvchap_budget AS FLOAT64) AS ecomRevActMktSLppvchap_budget,
      CAST(ecomRevActMktSLppvart_budget AS FLOAT64) AS ecomRevActMktSLppvart_budget,
      CAST(ecomRevActMktSLsubs_budget AS FLOAT64) AS ecomRevActMktSLsubs_budget,
      CAST(mqlJrnl_budget AS FLOAT64) AS mqlJrnl_budget,
      CAST(mqlBks_budget AS FLOAT64) AS mqlBks_budget,
      CAST(mqlDas_budget AS FLOAT64) AS mqlDas_budget,
      CAST(mqlIs_budget AS FLOAT64) AS mqlIs_budget,
      CAST(mqlRevRenewal_budget AS FLOAT64) AS mqlRevRenewal_budget,
      CAST(mqlRevUpsell_budget AS FLOAT64) AS mqlRevUpsell_budget,
      CAST(mqlRevOtherNew_budget AS FLOAT64) AS mqlRevOtherNew_budget,
      CAST(mqlRevOtherUpsell_budget AS FLOAT64) AS mqlRevOtherUpsell_budget,
      CAST(mqlRevOtherRenewal_budget AS FLOAT64) AS mqlRevOtherRenewal_budget,
      CAST(mqlRevOther_budget AS FLOAT64) AS mqlRevOther_budget,
      CAST(submMktInfl1Plus_budget AS FLOAT64) AS submMktInfl1Plus_budget,
      CAST(ImprintHubTraffic_budget AS FLOAT64) AS ImprintHubTraffic_budget

      FROM `snproject147bce46.2026_kpi_report.kpi_budget_sheet`
    )

  UNPIVOT

      (
        values for test IN
        (
        mqlRevBksECom_budget,
        mktIndirCost_budget,
        mktIndirCostPers_budget,
        mktIndirCostMktExp_budget,
        submMktInfl3Plus_budget,
        bookCntctSgnMkt_budget,
        fte_budget,
        b2cCntct_budget,
        b2cCntctOptin_budget,
        b2cAuthorsJrnl_budget,
        b2cAuthorsBks_budget,
        b2bCntct_budget,
        mqlRevJrnl_budget,
        mqlRevJrnlNew_budget,
        mqlRevJrnlUpsell_budget,
        mqlRevBks_budget,
        mqlRevBksNew_budget,
        mqlRevBksUpsell_budget,
        mqlRevBksRenewal_budget,
        mqlRevDas_budget,
        mqlRevDasNew_budget,
        mqlRevDasUpsell_budget,
        mqlRevDasRenewal_budget,
        mqlRevIs_budget,
        sal_budget,
        salJrnl_budget,
        salBks_budget,
        salDas_budget,
        salIs_budget,
        mql_budget,
        ecomRev_budget,
        ecomRevActMkt_budget,
        ecomRevActMktSLNAT_budget,
        ecomRevActMktB2BInst_budget,
        ecomRevSeo_budget,
        ecomRevSeoSLNAT_budget,
        ecomRevSeoSLbooksEbks_budget,
        ecomRevSeoSLppvchap_budget,
        ecomRevSeoSLppvart_budget,
        ecomRevSeoSLsubs_budget,
        ecomRevSeoB2BInst_budget,
        ecomRevActMktB2CsubsLTV_budget,
        ecomRevNatCarBDJ_budget,
        seo_budget,
        Traditionalseo_budget,
        AIseo_budget,
        sourcePageviews_budget,
        sourcePageviewsMktOptin_budget,
        productAWA_budget,
        productAWAeTOCnat_budget,
        productAWAeTOCspr_budget,
        productAWASourceSubs_budget,
        productAWABriefingSubs_budget,
        JrnlEngMkt_budget,
        JrnlEngMktOa_budget,
        elsMkt_budget,
        elsMktOa_budget,
        submitClickMkt_budget,
        submitClickMktOa_budget,
        submMktInflCol1Plus_budget,
        submMktInfl3PlusOa_budget,
        submMktInfl3PlusOtr_budget,
        powerAuthors_budget,
        submMktGen_budget,
        submMktGenOa_budget,
        oaAgreements_budget,
        oaAgreementsEligAuthorSubm_budget,
        bookLeadsAcc_budget,
        bookLeads_budget,
        ecomRevActMktSLbooksEbks_budget,
        ecomRevActMktSLppvchap_budget,
        ecomRevActMktSLppvart_budget,
        ecomRevActMktSLsubs_budget,
        mqlJrnl_budget,
        mqlBks_budget,
        mqlDas_budget,
        mqlIs_budget,
        mqlRevRenewal_budget,
        mqlRevUpsell_budget,
        mqlRevOtherNew_budget,
        mqlRevOtherUpsell_budget,
        mqlRevOtherRenewal_budget,
        mqlRevOther_budget,
        submMktInfl1Plus_budget,
        ImprintHubTraffic_budget
        )
      )

  )
)

GROUP BY month, kpi
ORDER BY month ASC

Query is running:   0%|          |

""


##forecast PIVOT


In [ ]:
%%bigquery --project  snproject147bce46

create or replace view `snproject147bce46.2026_kpi_report.pivot_forecast`

as

SELECT
Parse_date("%Y-%m-%d", Concat("2026-", month, "-1")) AS date,
kpi,
Sum(forecast) AS forecast

FROM
(
  SELECT
  month,
  kpi,
  CASE WHEN dimension = "forecast" THEN values END AS forecast

  FROM
  (
    SELECT
    month,
    values,
    Split(test, "_")[offset(0)] as kpi,
    split(test, "_")[offset(1)] as dimension

    FROM

    (
      SELECT
      extract(month from date) as month,
      CAST(mqlRevBksECom_forecast AS FLOAT64) AS mqlRevBksECom_forecast,
      CAST(mktIndirCost_forecast AS FLOAT64) AS mktIndirCost_forecast,
      CAST(mktIndirCostPers_forecast AS FLOAT64) AS mktIndirCostPers_forecast,
      CAST(mktIndirCostMktExp_forecast AS FLOAT64) AS mktIndirCostMktExp_forecast,
      CAST(submMktInfl3Plus_forecast AS FLOAT64) AS submMktInfl3Plus_forecast,
      CAST(bookCntctSgnMkt_forecast AS FLOAT64) AS bookCntctSgnMkt_forecast,
      CAST(fte_forecast AS FLOAT64) AS fte_forecast,
      CAST(b2cCntct_forecast AS FLOAT64) AS b2cCntct_forecast,
      CAST(b2cCntctOptin_forecast AS FLOAT64) AS b2cCntctOptin_forecast,
      CAST(b2cAuthorsJrnl_forecast AS FLOAT64) AS b2cAuthorsJrnl_forecast,
      CAST(b2cAuthorsBks_forecast AS FLOAT64) AS b2cAuthorsBks_forecast,
      CAST(b2bCntct_forecast AS FLOAT64) AS b2bCntct_forecast,
      CAST(mqlRevJrnl_forecast AS FLOAT64) AS mqlRevJrnl_forecast,
      CAST(mqlRevJrnlNew_forecast AS FLOAT64) AS mqlRevJrnlNew_forecast,
      CAST(mqlRevJrnlUpsell_forecast AS FLOAT64) AS mqlRevJrnlUpsell_forecast,
      CAST(mqlRevBks_forecast AS FLOAT64) AS mqlRevBks_forecast,
      CAST(mqlRevBksNew_forecast AS FLOAT64) AS mqlRevBksNew_forecast,
      CAST(mqlRevBksUpsell_forecast AS FLOAT64) AS mqlRevBksUpsell_forecast,
      CAST(mqlRevBksRenewal_forecast AS FLOAT64) AS mqlRevBksRenewal_forecast,
      CAST(mqlRevDas_forecast AS FLOAT64) AS mqlRevDas_forecast,
      CAST(mqlRevDasNew_forecast AS FLOAT64) AS mqlRevDasNew_forecast,
      CAST(mqlRevDasUpsell_forecast AS FLOAT64) AS mqlRevDasUpsell_forecast,
      CAST(mqlRevDasRenewal_forecast AS FLOAT64) AS mqlRevDasRenewal_forecast,
      CAST(mqlRevIs_forecast AS FLOAT64) AS mqlRevIs_forecast,
      CAST(sal_forecast AS FLOAT64) AS sal_forecast,
      CAST(salJrnl_forecast AS FLOAT64) AS salJrnl_forecast,
      CAST(salBks_forecast AS FLOAT64) AS salBks_forecast,
      CAST(salDas_forecast AS FLOAT64) AS salDas_forecast,
      CAST(salIs_forecast AS FLOAT64) AS salIs_forecast,
      CAST(mql_forecast AS FLOAT64) AS mql_forecast,
      CAST(ecomRev_forecast AS FLOAT64) AS ecomRev_forecast,
      CAST(ecomRevActMkt_forecast AS FLOAT64) AS ecomRevActMkt_forecast,
      CAST(ecomRevActMktSLNAT_forecast AS FLOAT64) AS ecomRevActMktSLNAT_forecast,
      CAST(ecomRevActMktB2BInst_forecast AS FLOAT64) AS ecomRevActMktB2BInst_forecast,
      CAST(ecomRevSeo_forecast AS FLOAT64) AS ecomRevSeo_forecast,
      CAST(ecomRevSeoSLNAT_forecast AS FLOAT64) AS ecomRevSeoSLNAT_forecast,
      CAST(ecomRevSeoSLbooksEbks_forecast AS FLOAT64) AS ecomRevSeoSLbooksEbks_forecast,
      CAST(ecomRevSeoSLppvchap_forecast AS FLOAT64) AS ecomRevSeoSLppvchap_forecast,
      CAST(ecomRevSeoSLppvart_forecast AS FLOAT64) AS ecomRevSeoSLppvart_forecast,
      CAST(ecomRevSeoSLsubs_forecast AS FLOAT64) AS ecomRevSeoSLsubs_forecast,
      CAST(ecomRevSeoB2BInst_forecast AS FLOAT64) AS ecomRevSeoB2BInst_forecast,
      CAST(ecomRevActMktB2CsubsLTV_forecast AS FLOAT64) AS ecomRevActMktB2CsubsLTV_forecast,
      CAST(ecomRevNatCarBDJ_forecast AS FLOAT64) AS ecomRevNatCarBDJ_forecast,
      CAST(seo_forecast AS FLOAT64) AS seo_forecast,
      CAST(Traditionalseo_forecast AS FLOAT64) AS Traditionalseo_forecast,
      CAST(AIseo_forecast AS FLOAT64) AS AIseo_forecast,
      CAST(sourcePageviews_forecast AS FLOAT64) AS sourcePageviews_forecast,
      CAST(sourcePageviewsMktOptin_forecast AS FLOAT64) AS sourcePageviewsMktOptin_forecast,
      CAST(productAWA_forecast AS FLOAT64) AS productAWA_forecast,
      CAST(productAWAeTOCnat_forecast AS FLOAT64) AS productAWAeTOCnat_forecast,
      CAST(productAWAeTOCspr_forecast AS FLOAT64) AS productAWAeTOCspr_forecast,
      CAST(productAWASourceSubs_forecast AS FLOAT64) AS productAWASourceSubs_forecast,
      CAST(productAWABriefingSubs_forecast AS FLOAT64) AS productAWABriefingSubs_forecast,
      CAST(JrnlEngMkt_forecast AS FLOAT64) AS JrnlEngMkt_forecast,
      CAST(JrnlEngMktOa_forecast AS FLOAT64) AS JrnlEngMktOa_forecast,
      CAST(elsMkt_forecast AS FLOAT64) AS elsMkt_forecast,
      CAST(elsMktOa_forecast AS FLOAT64) AS elsMktOa_forecast,
      CAST(submitClickMkt_forecast AS FLOAT64) AS submitClickMkt_forecast,
      CAST(submitClickMktOa_forecast AS FLOAT64) AS submitClickMktOa_forecast,
      CAST(submMktInflCol1Plus_forecast AS FLOAT64) AS submMktInflCol1Plus_forecast,
      CAST(submMktInfl3PlusOa_forecast AS FLOAT64) AS submMktInfl3PlusOa_forecast,
      CAST(submMktInfl3PlusOtr_forecast AS FLOAT64) AS submMktInfl3PlusOtr_forecast,
      CAST(powerAuthors_forecast AS FLOAT64) AS powerAuthors_forecast,
      CAST(submMktGen_forecast AS FLOAT64) AS submMktGen_forecast,
      CAST(submMktGenOa_forecast AS FLOAT64) AS submMktGenOa_forecast,
      CAST(oaAgreements_forecast AS FLOAT64) AS oaAgreements_forecast,
      CAST(oaAgreementsEligAuthorSubm_forecast AS FLOAT64) AS oaAgreementsEligAuthorSubm_forecast,
      CAST(bookLeadsAcc_forecast AS FLOAT64) AS bookLeadsAcc_forecast,
      CAST(bookLeads_forecast AS FLOAT64) AS bookLeads_forecast,
      CAST(ecomRevActMktSLbooksEbks_forecast AS FLOAT64) AS ecomRevActMktSLbooksEbks_forecast,
      CAST(ecomRevActMktSLppvchap_forecast AS FLOAT64) AS ecomRevActMktSLppvchap_forecast,
      CAST(ecomRevActMktSLppvart_forecast AS FLOAT64) AS ecomRevActMktSLppvart_forecast,
      CAST(ecomRevActMktSLsubs_forecast AS FLOAT64) AS ecomRevActMktSLsubs_forecast,
      CAST(mqlJrnl_forecast AS FLOAT64) AS mqlJrnl_forecast,
      CAST(mqlBks_forecast AS FLOAT64) AS mqlBks_forecast,
      CAST(mqlDas_forecast AS FLOAT64) AS mqlDas_forecast,
      CAST(mqlIs_forecast AS FLOAT64) AS mqlIs_forecast,
      CAST(mqlRevRenewal_forecast AS FLOAT64) AS mqlRevRenewal_forecast,
      CAST(mqlRevUpsell_forecast AS FLOAT64) AS mqlRevUpsell_forecast,
      CAST(mqlRevOtherNew_forecast AS FLOAT64) AS mqlRevOtherNew_forecast,
      CAST(mqlRevOtherUpsell_forecast AS FLOAT64) AS mqlRevOtherUpsell_forecast,
      CAST(mqlRevOtherRenewal_forecast AS FLOAT64) AS mqlRevOtherRenewal_forecast,
      CAST(mqlRevOther_forecast AS FLOAT64) AS mqlRevOther_forecast,
      CAST(submMktInfl1Plus_forecast AS FLOAT64) AS submMktInfl1Plus_forecast,
      CAST(ImprintHubTraffic_forecast AS FLOAT64) AS ImprintHubTraffic_forecast,

      FROM `snproject147bce46.2026_kpi_report.kpi_forecast_sheet`
    )

  UNPIVOT

      (
        values for test IN
        (
        mqlRevBksECom_forecast,
        mktIndirCost_forecast,
        mktIndirCostPers_forecast,
        mktIndirCostMktExp_forecast,
        submMktInfl3Plus_forecast,
        bookCntctSgnMkt_forecast,
        fte_forecast,
        b2cCntct_forecast,
        b2cCntctOptin_forecast,
        b2cAuthorsJrnl_forecast,
        b2cAuthorsBks_forecast,
        b2bCntct_forecast,
        mqlRevJrnl_forecast,
        mqlRevJrnlNew_forecast,
        mqlRevJrnlUpsell_forecast,
        mqlRevBks_forecast,
        mqlRevBksNew_forecast,
        mqlRevBksUpsell_forecast,
        mqlRevBksRenewal_forecast,
        mqlRevDas_forecast,
        mqlRevDasNew_forecast,
        mqlRevDasUpsell_forecast,
        mqlRevDasRenewal_forecast,
        mqlRevIs_forecast,
        sal_forecast,
        salJrnl_forecast,
        salBks_forecast,
        salDas_forecast,
        salIs_forecast,
        mql_forecast,
        ecomRev_forecast,
        ecomRevActMkt_forecast,
        ecomRevActMktSLNAT_forecast,
        ecomRevActMktB2BInst_forecast,
        ecomRevSeo_forecast,
        ecomRevSeoSLNAT_forecast,
        ecomRevSeoSLbooksEbks_forecast,
        ecomRevSeoSLppvchap_forecast,
        ecomRevSeoSLppvart_forecast,
        ecomRevSeoSLsubs_forecast,
        ecomRevSeoB2BInst_forecast,
        ecomRevActMktB2CsubsLTV_forecast,
        ecomRevNatCarBDJ_forecast,
        seo_forecast,
        Traditionalseo_forecast,
        AIseo_forecast,
        sourcePageviews_forecast,
        sourcePageviewsMktOptin_forecast,
        productAWA_forecast,
        productAWAeTOCnat_forecast,
        productAWAeTOCspr_forecast,
        productAWASourceSubs_forecast,
        productAWABriefingSubs_forecast,
        JrnlEngMkt_forecast,
        JrnlEngMktOa_forecast,
        elsMkt_forecast,
        elsMktOa_forecast,
        submitClickMkt_forecast,
        submitClickMktOa_forecast,
        submMktInflCol1Plus_forecast,
        submMktInfl3PlusOa_forecast,
        submMktInfl3PlusOtr_forecast,
        powerAuthors_forecast,
        submMktGen_forecast,
        submMktGenOa_forecast,
        oaAgreements_forecast,
        oaAgreementsEligAuthorSubm_forecast,
        bookLeadsAcc_forecast,
        bookLeads_forecast,
        ecomRevActMktSLbooksEbks_forecast,
        ecomRevActMktSLppvchap_forecast,
        ecomRevActMktSLppvart_forecast,
        ecomRevActMktSLsubs_forecast,
        mqlJrnl_forecast,
        mqlBks_forecast,
        mqlDas_forecast,
        mqlIs_forecast,
        mqlRevRenewal_forecast,
        mqlRevUpsell_forecast,
        mqlRevOtherNew_forecast,
        mqlRevOtherUpsell_forecast,
        mqlRevOtherRenewal_forecast,
        mqlRevOther_forecast,
        submMktInfl1Plus_forecast,
        ImprintHubTraffic_forecast
        )
      )

  )
)

GROUP BY month, kpi
ORDER BY month ASC

Query is running:   0%|          |

""


##Final Table - JOIN budget and forecast with actuals and py

In [ ]:
%%bigquery --project  snproject147bce46
create or replace view `snproject147bce46.2026_kpi_report.pivot_actuals_py_budget_forecast`

as

SELECT

date,
kpi,
ordernumber,
IFNULL(actual, 0.00) as actual,
IFNULL(py, 0.00) as py,
IFNULL(budget, 0.00) as budget,
IFNULL(forecast, 0.00) as forecast

FROM

(
  SELECT *
  FROM `snproject147bce46.2026_kpi_report.pivot_actuals_py_order`

  FULL JOIN `snproject147bce46.2026_kpi_report.pivot_budget`
  USING (date, kpi)

  FULL JOIN `snproject147bce46.2026_kpi_report.pivot_forecast`
  USING (date, kpi)
)
ORDER BY date asc, ordernumber asc

Query is running:   0%|          |

""


#Code to restore the data

In [ ]:
#Code to restore the data
#Change the table name
%%bigquery --project  snproject147bce46

create or replace table `snproject147bce46.2025_kpi_report.researcher_mkt` as
SELECT *
FROM `snproject147bce46.2025_kpi_report.researcher_mkt`
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)

#Code to delete rows

In [ ]:
%%bigquery --project  snproject147bce46

DELETE `snproject147bce46.2026_kpi_report.ecom`
WHERE date > '2025-12-31'

Query is running:   0%|          |

""


#Code to update particular columns

In [ ]:
UPDATE `snproject147bce46.2025_kpi_report.mql_jbd`
SET
  salJrnl = src.salJrnl,
  salBks  = src.salBks,
  salDas  = src.salDas
FROM (
SELECT
 FORMAT_DATE("%Y-%m", CAST(FORMAT_DATETIME("%F", created_date) AS date)) AS date,
 SUM(CASE WHEN high_level_product = 'Journals' THEN total_sal END) AS salJrnl,
 SUM(CASE WHEN high_level_product = 'eBooks' THEN total_sal END) AS salBks,
 SUM(CASE WHEN high_level_product = 'Databases' THEN total_sal END) AS salDas,
FROM `snproject-cbb9ca25.hex_final_tables.b2b_leads_all`
-- WHERE created_year > 2023
where created_year > 2024
AND high_level_product IS NOT NULL
GROUP BY 1
ORDER BY 1 ASC
) src
WHERE `snproject147bce46.2025_kpi_report.mql_jbd`.date = src.date
  AND LEFT(`snproject147bce46.2025_kpi_report.mql_jbd`.date, 4) = '2025';

#------------- Not Required -------------

##Not required - B2B - RM scorecard

##Financial KPIs

In [ ]:
%%bigquery --project  snproject147bce46

create or replace view `snproject147bce46.2025_kpi_report.ops_B2B_Rev`

as

SELECT
date,
case
      when kpi = "mqlRev" then 1
      when kpi = "mqlRevJrnl" then 2
      when kpi = "mqlRevBks" then 3
      when kpi = "mqlRevDas" then 4
      when kpi = "mqlRevNRI" then 5
      when kpi = "mqlRevProtocolsIo" then 6
      when kpi = "mqlRevIs" then 7
      when kpi = "mqlRevTrainingServices" then 8

end as ordernumber,
kpi,
case
      when kpi = "mqlRev" then "Marketing Lead influenced revenue"
      when kpi = "mqlRevJrnl" then "Journals Revenue"
      when kpi = "mqlRevBks" then "Books Revenue"
      when kpi = "mqlRevDas" then "DAS Revenue"
      when kpi = "mqlRevNRI" then "NRI Revenue"
      when kpi = "mqlRevProtocolsIo" then "Protocols.io Revenue"
      when kpi = "mqlRevIs" then "Impact Solutions Revenue"
      when kpi = "mqlRevTrainingServices" then "t/o. Training Services Revenue"

end as kpi_human,
actual,
budget,
forecast,
py

from `snproject147bce46.2025_kpi_report.pivot_public_human_actuals_py_budget_forecast`
where kpi in ("mqlRev",
              "mqlRevJrnl",
              "mqlRevBks",
              "mqlRevDas",
              "mqlRevNRI",
              "mqlRevIs",
              "mqlRevProtocolsIo",
              "mqlRevTrainingServices")

##MQL and SAL

In [ ]:
%%bigquery --project  snproject147bce46

create or replace view `snproject147bce46.2025_kpi_report.ops_B2B`

as

SELECT
date,
case
      when kpi = "mql"  then 1
      when kpi = "mqlJrnl"  then 2
      when kpi = "mqlBks"  then 3
      when kpi = "mqlDas"  then 4
      when kpi = "mqlNRI"  then 5
      when kpi = "mqlProtocolsIo" then 6
      when kpi = "mqlIs"  then 7
      when kpi = "sal"  then 8
      when kpi = "salJrnl"  then 9
      when kpi = "salBks"  then 10
      when kpi = "salDas"  then 11
      when kpi = "salNRI"  then 12
      when kpi = "salProtocolsIo" then 13
      when kpi = "salIs"  then 14

end as ordernumber,
kpi,
case
      when kpi = "sal" then "# Sales Accepted Leads"
      when kpi = "salJrnl" then "Journals SAL"
      when kpi = "salBks" then "Books SAL"
      when kpi = "salDas" then "DAS SAL"
      when kpi = "salNRI" then "NRI SAL"
      when kpi = "salProtocolsIo" then "Protocols.io SAL"
      when kpi = "salIs" then "Impact Solutions SAL"
      when kpi = "mql" then "# of MQL3"
      when kpi = "mqlJrnl" then "Journals MQL3"
      when kpi = "mqlBks" then "Books MQL3"
      when kpi = "mqlDas" then "DAS MQL3"
      when kpi = "mqlProtocolsIo" then "Protocols.io MQL3"
      when kpi = "mqlIs" then "Impact Solutions MQL3"
      when kpi = "mqlNRI" then "NRI MQL3"
end as kpi_human,
actual,
budget,
forecast,
py

from `snproject147bce46.2025_kpi_report.pivot_public_human_actuals_py_budget_forecast`
where kpi in ("mql",
"mqlJrnl",
"mqlBks",
"mqlDas",
"mqlNRI",
"mqlIs",
"mqlProtocolsIo",
"sal",
"salJrnl",
"salBks",
"salDas",
"salNRI",
"salIs",
"salProtocolsIo")

##SAL %

In [ ]:
%%bigquery --project  snproject147bce46

create or replace view `snproject147bce46.2025_kpi_report.ops_B2B_SAL`

as

select

date,
case
when kpi = "sal" or kpi = "mql" then 1
  when kpi = "salJrnl" or kpi = "mqlJrnl"  then 2
  when kpi = "salBks" or kpi = "mqlBks"  then 3
  when kpi = "salDas" or kpi = "mqlDas"  then 4
  when kpi = "salProtocolsIo" or kpi = "mqlProtocolsIo"  then 5
  when kpi = "salIs" or kpi = "mqlIs"  then 6
end as ordernumber,

case
when kpi = "sal" or kpi = "mql" then "MQL 3 Sales Acceptance Rate (%)"
  when kpi = "salJrnl" or kpi = "mqlJrnl"  then "Journals SAL%"
  when kpi = "salBks" or kpi = "mqlBks"  then "Books SAL%"
  when kpi = "salDas" or kpi = "mqlDas"  then "DAS SAL%"
  when kpi = "salProtocolsIo" or kpi = "mqlProtocolsIo"  then "Protocols.io SAL%"
  when kpi = "salIs" or kpi = "mqlIs"  then "Impact Solutions SAL%"
end as kpi_human,

case
when kpi="sal" then actual
when kpi="salJrnl" then actual
when kpi="salBks" then actual
when kpi="salDas" then actual
when kpi="salProtocolsIo" then actual
when kpi="salIs" then actual
end as actual_sal,

case
when kpi="sal" then budget
when kpi="salJrnl" then budget
when kpi="salBks" then budget
when kpi="salDas" then budget
when kpi="salProtocolsIo" then budget
when kpi="salIs" then budget
end as budget_sal,

case
when kpi="sal" then forecast
when kpi="salJrnl" then forecast
when kpi="salBks" then forecast
when kpi="salDas" then forecast
when kpi="salProtocolsIo" then forecast
when kpi="salIs" then forecast
end as forecast_sal,

case
when kpi="sal" then py
when kpi="salJrnl" then py
when kpi="salBks" then py
when kpi="salDas" then py
when kpi="salProtocolsIo" then py
when kpi="salIs" then py
end as py_sal,


case
when kpi="mql" then actual
when kpi="mqlJrnl" then actual
when kpi="mqlBks" then actual
when kpi="mqlDas" then actual
when kpi="mqlProtocolsIo" then actual
when kpi="mqlIs" then actual
end as actual_mql,

case
when kpi="mql" then budget
when kpi="mqlJrnl" then budget
when kpi="mqlBks" then budget
when kpi="mqlDas" then budget
when kpi="mqlProtocolsIo" then budget
when kpi="mqlIs" then budget
end as budget_mql,

case
when kpi="mql" then forecast
when kpi="mqlJrnl" then forecast
when kpi="mqlBks" then forecast
when kpi="mqlDas" then forecast
when kpi="mqlProtocolsIo" then forecast
when kpi="mqlIs" then forecast
end as forecast_mql,

case
when kpi="mql" then py
when kpi="mqlJrnl" then py
when kpi="mqlBks"  then py
when kpi="mqlDas"  then py
when kpi="mqlProtocolsIo"  then py
when kpi="mqlIs" then py
end as py_mql


from `snproject147bce46.2025_kpi_report.pivot_public_human_actuals_py_budget_forecast`



##hide sensitive KPIs

####Scheduled Query - [link](https://console.cloud.google.com/bigquery/scheduled-queries/locations/europe/configs/67cafd0f-0000-259c-8cfa-f4f5e80f07c0/runs?invt=AbtxAg&project=snproject147bce46)

In [ ]:
%%bigquery --project  snproject147bce46
create or replace table `snproject147bce46.2025_kpi_report.pivot_public_human_actuals_py_budget_forecast`

as

SELECT
date,
ordernumber,
kpi,
case when kpi = "mktIndirCostMktExp" then "Marketing expense" else kpi_human end as kpi_human,
actual,
budget,
forecast,
py

FROM `snproject147bce46.2025_kpi_report.pivot_human_actuals_py_budget_forecast`

where kpi not in ("fte", "mktIndirCost", "mktIndirCostPers")

Query is running:   0%|          |

""


##add human name

In [ ]:
%%bigquery --project  snproject147bce46


create or replace view `snproject147bce46.2025_kpi_report.pivot_human_actuals_py_budget_forecast`

as

SELECT
date,
ordernumber,
kpi,
case
when kpi = 'mqlRevBksECom' then "Marketing Influenced/ generated new / upsell book revenue [m]"
when kpi = 'mqlRev' then "Marketing Lead influenced revenue [m]"
when kpi = 'mqlRevNew' then "t/o New Business"
when kpi = 'mqlRevUpsell' then "t/o Upsell Business"
when kpi = 'mqlRevRenewal' then "t/o Renewals"
when kpi = 'ecomRevActMkt' then "eCommerce revenue paid marketing [m]"
when kpi = 'ecomRevActMktSL' then "t/o B2C SpringerLink"
when kpi = 'ecomRevActMktSLNAT' then "t/o B2C SpringerLink and Nature"
when kpi = 'ecomRevActMktSLppvchap' then "t/o B2C SpringerLink PPV Chapter"
when kpi = 'ecomRevActMktB2Csubs' then "t/o B2C Subs (Nature+)"
when kpi = 'ecomRevActMktB2CsubsLTV' then "t/o B2C subs (Live Time Value)"
when kpi = 'ecomRevActMktB2BInst' then "t/o B2B institutional"
when kpi = 'ecomRevSeo' then "eCommerce revenue SEO [m]"
when kpi = 'ecomRevSeoSL' then "t/o B2C SpringerLink"
when kpi = 'ecomRevSeoSLNAT' then "t/o B2C SpringerLink and Nature"
when kpi = 'ecomRevSeoSLppvchap' then "t/o B2C SpringerLink PPV Chapter"
when kpi = 'ecomRevSeoB2Csubs' then "t/o B2C Subs (Nature+)"
when kpi = 'ecomRevSeoB2CsubsLTV' then "t/o B2C subs (Live Time Value)"
when kpi = 'ecomRevSeoB2BInst' then "t/o B2B institutional"
when kpi = 'ecomRevNatCarBDJ' then "eCommerce revenue Nature Careers and British Dental Jobs [m]"
when kpi = 'mktIndirCost' then "Marketing indirect cost [m]"
when kpi = 'mktIndirCostPers' then "thereof Personnel cost"
when kpi = 'mktIndirCostMktExp' then "thereof Marketing expense"
when kpi = 'submMktInfl3Plus' then "# Marketing-influenced submissions (3+ interactions)"
when kpi = 'submMktGen' then "Marketing generated submissions"
when kpi = 'bookCntctSgnMkt' then "Marketing influenced book contracts"
when kpi = 'fte' then "FTEs"
when kpi = 'b2cCntct' then "# B2C contacts [k]"
when kpi = 'b2cCntctOptin' then "t/o opt-ins"
when kpi = 'b2cAuthors' then "t/o author profiles"
when kpi = 'b2bCntct' then "# B2B contacts [k]"
when kpi = 'mqlRevJrnl' then "mqlRevJrnl"
when kpi = 'mqlRevJrnlNew' then "mqlRevJrnlNew"
when kpi = 'mqlRevJrnlUpsell' then "mqlRevJrnlUpsell"
when kpi = 'mqlRevBks' then "mqlRevBks"
when kpi = 'mqlRevBksNew' then "mqlRevBksNew"
when kpi = 'mqlRevBksUpsell' then "mqlRevBksUpsell"
when kpi = 'mqlRevBksRenewal' then "mqlRevBksRenewal"
when kpi = 'mqlRevDas' then "mqlRevDas"
when kpi = 'mqlRevDasNew' then "mqlRevDasNew"
when kpi = 'mqlRevDasUpsell' then "mqlRevDasUpsell"
when kpi = 'mqlRevDasRenewal' then "mqlRevDasRenewal"
when kpi = 'mqlRevNRI' then "NRI Marketing Lead influenced revenue"
when kpi = 'mqlRevProtocolsIo' then "Protocols.io Marketing Lead influenced revenue"
when kpi = 'mqlRevIs' then "IS (incl Training Services) Marketing"
when kpi = 'sal' then "# Sales Accepted Leads"
when kpi = 'salJrnl' then "Journals SAL"
when kpi = 'salBks' then "Books SAL"
when kpi = 'salDas' then "DAS SAL"
when kpi = 'salNRI' then "NRI SAL"
when kpi = 'salProtocolsIo' then "Protocols.io SAL"
when kpi = 'salIs' then "Impact Solutions SAL"
when kpi = 'mql' then "# of MQL3 "
when kpi = 'seo' then "Total SEO traffic (GA sessions) [m]"
when kpi = 'Traditionalseo' then "Traditional SEO traffic (GA sessions) [m]"
when kpi = 'AIseo' then "AI SEO traffic (GA sessions) [m]"
when kpi = 'sourcePageviews' then "# of total page views to The Source blog"
when kpi = 'sourcePageviewsMktOptin' then "of which from marketing-opt in contacts"
when kpi = 'SourceReadership' then "Readership metric on The Source"
when kpi = 'JrnlEngMkt' then "# of Journal Engagement Conversions (Marketing)"
when kpi = 'JrnlEngMktOa' then "of which: Open Access journals"
when kpi = 'engSessionsJrnlMkt' then "# of Engaged Sessions to journals (Marketing)"
when kpi = 'engSessionsJrnlMktOa' then "of which: Open Access journals"
when kpi = 'elsMkt' then "# of ELS Conversions (Marketing)"
when kpi = 'elsMktOa' then "of which: Open Access journals"
when kpi = 'submitClickMkt' then "of which: Marketing-driven submit clicks"
when kpi = 'submitClickMktOa' then "of which: Open Access journals"
when kpi = 'submMktInfl1Plus' then "# Marketing-influenced submissions (1+ interactions)"
when kpi = 'submMktInflCol1Plus' then "t/o Collections submissions (1+ interactions)"
when kpi = 'submMktInfl3PlusOa' then "of which: Open Access journals (3+ interactions)"
when kpi = 'powerAuthors' then "# submissions from Power Authors"
when kpi = 'submMktGenOa' then "of which: Open Access journals"
when kpi = 'oaAgreements' then "OA Agreements - % TA"
when kpi = 'oaAgreementsEligAuthorSubm' then "OA Agreements - # subs from eligible authors"
when kpi = 'natureBriefingSgn' then "Nature Briefing Sign-ups"
when kpi = 'bookLeadsAcc' then "Book author leads editorial acceptances"
when kpi = 'bookLeads' then "Book author leads"
when kpi = 'mqlRevTrainingServices' then "mqlRevTrainingServices"
when kpi = 'mqlIs' then "mqlIs"
when kpi = 'mqlNRI' then "mqlNRI"
when kpi = 'mqlProtocolsIo' then "mqlProtocolsIo"
when kpi = 'mqlJrnl' then "mqlJrnl"
when kpi = 'mqlBks' then "mqlBks"
when kpi = 'mqlDas' then "mqlDas"

    end as kpi_human,
actual,
budget,
forecast,
py

FROM `snproject147bce46.2025_kpi_report.pivot_actuals_py_budget_forecast`

Query is running:   0%|          |

""


#BRM Dashboard
Researcher and ecommerce KPIs

In [ ]:
create or replace table `snproject147bce46.2025_kpi_report.dashboard_b2c_ecom`
as

select t1.* except(month,year,seo,elsMkt,elsMktOa,JrnlEngMkt,JrnlEngMktOa,submitClickMkt,submitClickMktOa),t2.* except(date),
IFNULL(t4.mqlRevBksNew,0.0) + IFNULL(t4.mqlRevBksUpsell,0.0) + IFNULL(ecomRevActMktSL,0.0) + IFNULL(ecomRevSeoSL,0.0)+ ecomRevSeoB2BInst + ecomRevActMktB2BInst as MktGenInflBookRev,
mqlRevBksNew, mqlRevBksUpsell,
t5.* except(date),
t6.* except(date),

case
when month = 1 and year = 2024 then 49028110.0
when month = 2 and year = 2024 then 47909095.0
when month = 3 and year = 2024 then 51274367.0
when month = 4 and year = 2024 then 54787769.0
when month = 5 and year = 2024 then 50308036.0
when month = 6 and year = 2024 then 40350154.0
else seo
end as seo,

case
when month = 1 and year = 2024 then 116736.0
when month = 2 and year = 2024 then 107842.0
when month = 3 and year = 2024 then 135068.0
when month = 4 and year = 2024 then 181262.0
when month = 5 and year = 2024 then 218471.0
when month = 6 and year = 2024 then 240477.0
else submitClickMkt
end as submitClickMkt,

case
when month = 1 and year = 2024 then 56637.0
when month = 2 and year = 2024 then 58384.0
when month = 3 and year = 2024 then 65175.0
when month = 4 and year = 2024 then 105942.0
when month = 5 and year = 2024 then 134979.0
when month = 6 and year = 2024 then 166337.0
else submitClickMktOa
end as submitClickMktOa,

case
when month = 1 and year = 2024 then 0.0
when month = 2 and year = 2024 then 0.0
when month = 3 and year = 2024 then 0.0
else elsMkt
end as elsMkt,

case
when month = 1 and year = 2024 then 0.0
when month = 2 and year = 2024 then 0.0
when month = 3 and year = 2024 then 0.0
else elsMktOa
end as elsMktOa,

##journal_engagement event was modified to fire when user reaches 50% scroll on journal pages
##The event was modified on 2025-04-16
case
when month = 1 and year = 2024 then 0.0
when month = 2 and year = 2024 then 0.0
when month = 3 and year = 2024 then 0.0
when month = 4 and year = 2024 then 0.0
when month = 5 and year = 2024 then 0.0
when month = 6 and year = 2024 then 0.0
when month = 7 and year = 2024 then 0.0
when month = 8 and year = 2024 then 0.0
when month = 9 and year = 2024 then 0.0
when month = 10 and year = 2024 then 0.0
when month = 11 and year = 2024 then 0.0
when month = 12 and year = 2024 then 0.0
when month = 1 and year = 2025 then 0.0
when month = 2 and year = 2025 then 0.0
when month = 3 and year = 2025 then 0.0
when month = 4 and year = 2025 then 0.0
else JrnlEngMkt
end as JrnlEngMkt,

case
when month = 1 and year = 2024 then 0.0
when month = 2 and year = 2024 then 0.0
when month = 3 and year = 2024 then 0.0
when month = 4 and year = 2024 then 0.0
when month = 5 and year = 2024 then 0.0
when month = 6 and year = 2024 then 0.0
when month = 7 and year = 2024 then 0.0
when month = 8 and year = 2024 then 0.0
when month = 9 and year = 2024 then 0.0
when month = 10 and year = 2024 then 0.0
when month = 11 and year = 2024 then 0.0
when month = 12 and year = 2024 then 0.0
when month = 1 and year = 2025 then 0.0
when month = 2 and year = 2025 then 0.0
when month = 3 and year = 2025 then 0.0
when month = 4 and year = 2025 then 0.0
else JrnlEngMktOa
end as JrnlEngMktOa,

  case
  when month = 1 and year = 2024 then 260000.4
  when month = 2 and year = 2024 then 225249.0
  when month = 3 and year = 2024 then 231185.8
  when month = 4 and year = 2024 then 182360.9
  when month = 5 and year = 2024 then 242248.9
  when month = 6 and year = 2024 then 197616.2
  else ecomRevActMktSL
  end as ecomRevActMktSL,

  case
  when month = 1 and year = 2024 then 497440.9
  when month = 2 and year = 2024 then 366547.4
  when month = 3 and year = 2024 then 342475.7
  when month = 4 and year = 2024 then 325127.4
  when month = 5 and year = 2024 then 351650.6
  when month = 6 and year = 2024 then 274486.4
  else ecomRevSeoSL
  end as ecomRevSeoSL,

  case
  when month = 1 and year = 2024 then 277981.6
  when month = 2 and year = 2024 then 236102.6
  when month = 3 and year = 2024 then 241859.2
  when month = 4 and year = 2024 then 192157.9
  when month = 5 and year = 2024 then 248815.4
  when month = 6 and year = 2024 then 199016.7
  else ecomRevActMktSL_NAT
  end as ecomRevActMktSL_NAT,

  case
  when month = 1 and year = 2024 then 544983.9
  when month = 2 and year = 2024 then 405819.3
  when month = 3 and year = 2024 then 378825.7
  when month = 4 and year = 2024 then 355163.3
  when month = 5 and year = 2024 then 377767.9
  when month = 6 and year = 2024 then 276220.9
  else ecomRevSeoSL_NAT
  end as ecomRevSeoSL_NAT,

  case
  when	month	=	1	and	year	=	2024	then	1013.5
  when	month	=	2	and	year	=	2024	then	1001.8
  when	month	=	3	and	year	=	2024	then	1058.4
  when	month	=	4	and	year	=	2024	then	1150.9
  when	month	=	5	and	year	=	2024	then	1221.4
  when	month	=	6	and	year	=	2024	then	1176.7
  else ecomRevActMktSL_ppv_chap
  end as ecomRevActMktSL_ppv_chap,

  case
  when	month	=	1	and	year	=	2024	then	57656.9
  when	month	=	2	and	year	=	2024	then	46577.6
  when	month	=	3	and	year	=	2024	then	46174.5
  when	month	=	4	and	year	=	2024	then	42758.3
  when	month	=	5	and	year	=	2024	then	32471.6
  when	month	=	6	and	year	=	2024	then	33138.8
  else ecomRevSeoSL_ppv_chap
  end as ecomRevSeoSL_ppv_chap,

  case
  when	month	=	1	and	year	=	2024	then	3665.0
  when	month	=	2	and	year	=	2024	then	2578.0
  when	month	=	3	and	year	=	2024	then	7536.0
  when	month	=	4	and	year	=	2024	then	3606.0
  when	month	=	5	and	year	=	2024	then	1060.0
  when	month	=	6	and	year	=	2024	then	6064.0
  else ecomRevActMktB2BInst
  end as ecomRevActMktB2BInst,

  case
  when	month	=	1	and	year	=	2024	then	2968.0
  when	month	=	2	and	year	=	2024	then	14478.0
  when	month	=	3	and	year	=	2024	then	18976.0
  when	month	=	4	and	year	=	2024	then	8266.0
  when	month	=	5	and	year	=	2024	then	14675.0
  when	month	=	6	and	year	=	2024	then	7305.0
  else ecomRevSeoB2BInst
  end as ecomRevSeoB2BInst

from
  (
    --seo and product awareness,consideration
    select format_date("%Y-%m", date) as date,  -- Corrected line
    EXTRACT(month FROM date) AS month,
    EXTRACT(year FROM date) AS year,
    IFNULL(sum(case when imprint is not null  and sub_channels = "seo" then sessions else 0 end),0) as seo,
    IFNULL(sum(case when research_marketing_activity is true then journal_engagement else 0 end),0) as JrnlEngMkt,
    IFNULL(sum(case when research_marketing_activity is true and oatype = "OA" then journal_engagement else 0 end),0) as JrnlEngMktOa,
    IFNULL(sum(case when research_marketing_activity is true then ELS else 0 end),0) as elsMkt,
    IFNULL(sum(case when research_marketing_activity is true and oatype = "OA" then ELS else 0 end),0) as elsMktOa,
    IFNULL(sum(case when research_marketing_activity is true then kpi_submit_click else 0 end),0) as SubmitClickMkt,
    IFNULL(sum(case when research_marketing_activity is true and oatype = "OA" then kpi_submit_click else 0 end),0) as SubmitClickMktOA

    from `ga4_ria_researcher_report.j7`
    where date between "2024-01-01" and "2025-04-30"
    group by 1,2,3
    order by 1
  )t1

  left join

  (
    --brand attract and engage
    SELECT * except (date),
    format_date("%Y-%m",date) as date,
    FROM `snproject147bce46.2025_kpi_report.kpi_dashboard_sheet`
  )t2

  on t1.date = t2.date

  left join

(
  SELECT a.*, IFNULL(b.ecomRevActMktB2BInst,0.0) as ecomRevActMktB2BInst,IFNULL(b.ecomRevSeoB2BInst,0.0) ecomRevSeoB2BInst
  from
  (
  SELECT
  date,
  SUM(IFNULL(ecomRevActMktSL,0)) as ecomRevActMktSL,
  SUM(IFNULL(ecomRevSeoSL,0)) as ecomRevSeoSL,

  SUM(IFNULL(ecomRevActMktSL_NAT,0)) as ecomRevActMktSL_NAT,
  SUM(IFNULL(ecomRevSeoSL_NAT,0)) as ecomRevSeoSL_NAT,

  SUM(IFNULL(ecomRevActMktSL_ppv_chap,0)) as ecomRevActMktSL_ppv_chap,
  SUM(IFNULL(ecomRevSeoSL_ppv_chap,0)) as ecomRevSeoSL_ppv_chap

  FROM
  (SELECT
  date, month, year,
  sum(case when exec_summary_areas = "of which: Active Marketing" and platform = 'springerlink' then revenue end) as ecomRevActMktSL,
  sum(case when exec_summary_areas = "of which: SEO" and platform = 'springerlink' then revenue end) as ecomRevSeoSL,

  sum(case when exec_summary_areas = "of which: Active Marketing" then revenue end) as ecomRevActMktSL_NAT,
  sum(case when exec_summary_areas = "of which: SEO" then revenue end) as ecomRevSeoSL_NAT,

  sum(case when exec_summary_areas = "of which: Active Marketing" and item_variant = "ppv-chapter" then revenue end) as ecomRevActMktSL_ppv_chap,
  sum(case when exec_summary_areas = "of which: SEO" and item_variant = "ppv-chapter" then revenue end) as ecomRevSeoSL_ppv_chap


  FROM
  (SELECT
      format_date("%Y-%m", date) as date,
      EXTRACT(month FROM date) AS month,
      EXTRACT(year FROM date) AS year,
      CASE WHEN active_marketing_incl_coupon_mistrack = true THEN "of which: Active Marketing"
      WHEN marmind_channels_details = 'seo' and active_marketing_incl_coupon_mistrack = false THEN "of which: SEO"
      ELSE "of which: Non Active Marketing"
      END exec_summary_areas,
      platform,
      item_variant,
      sum(ifnull(item_revenue,0)) as revenue

      FROM `snproject-cbb9ca25.ria_ecom_reporting.transactions_full`
      where date between  "2024-01-01" and "2025-04-30"
      -- where date between date_trunc(date_sub(current_date(), interval 1 month), month) and last_day(date_sub(current_date(), interval 1 month), month)
      and platform is not null #this table does not contain natureasia data, we do not have any data for NatureAsia

      GROUP BY date, active_marketing_incl_coupon_mistrack, marmind_channels_details, platform, item_variant, month, year

      ORDER BY date ASC
  )
  GROUP BY date, month, year
  ORDER BY date ASC
  )
  GROUP BY date,month, year
  ORDER BY date ASC
  )a
  left join
  (

WITH deduplicated AS (
  SELECT * EXCEPT(month, year),
  EXTRACT(MONTH FROM date) AS month, EXTRACT(YEAR FROM date) AS year,
  ROW_NUMBER() OVER (PARTITION BY transaction_id, item_id ORDER BY date ASC) AS unique_row_number
  FROM `snproject-cbb9ca25.hex_final_tables.b2b_sebs_final`
),
revenue AS (
  SELECT
    FORMAT_DATE("%Y-%m", date) AS date,
    month,
    year,
    SUM(CASE WHEN revenue_breakdown = '1. of which: Marketing' THEN FLOOR(item_revenue) END) AS ecomRevActMktB2BInst,
    SUM(CASE WHEN revenue_breakdown = '3. of which: SEO' THEN FLOOR(item_revenue) END) AS ecomRevSeoB2BInst
  FROM
    deduplicated
  WHERE date > '2023-12-31'
    AND unique_row_number = 1
  GROUP BY 1, 2, 3
  ORDER BY 1
)
select * except(month,year) from revenue

  )b
  on a.date = b.date
)t3

on t1.date = t3.date

left join

(
  WITH
b2b_sf_con_rev AS(
  SELECT
    "b2b_sf_con_rev" AS revenue_type,
    FORMAT_DATE("%Y-%m", opp_close_date) AS date,
    line_item_high_level_product,
    line_item_type_of_business,
    ifnull(ROUNd(SUM(line_item_total_price_conv),2),0) AS revenue
  FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_con_rev_final_v2`
  WHERE opp_close_date > '2023-12-31'
  GROUP BY 1,2,3,4
),

b2b_sf_non_sql_rev AS(
  SELECT
   "b2b_sf_non_sql_rev" AS revenue_type,
    FORMAT_DATE("%Y-%m", opp_close_date) AS date,
    line_item_high_level_product,
    line_item_type_of_business,
    ifnull(ROUND(SUM(line_item_total_price_conv),2),0) AS revenue
  FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_non_sql_rev_final_v2`
  WHERE opp_close_date > '2023-12-31'
  GROUP BY 1,2,3,4
),

b2b_sf_sql_rev AS(
  SELECT
    "b2b_sf_sql_rev" AS revenue_type,
    FORMAT_DATE("%Y-%m", opp_close_date) AS date,
    line_item_high_level_product,
    line_item_type_of_business,
    ifnull(ROUND(SUM(line_item_total_price_conv),2),0) AS revenue
  FROM `snproject-cbb9ca25.hex_final_tables.b2b_sf_sql_rev_final_v2`
  WHERE opp_close_date > '2023-12-31'
  GROUP BY 1,2,3,4
),

combined_data AS(
    SELECT * FROM b2b_sf_con_rev
    UNION ALL
    SELECT * FROM b2b_sf_sql_rev
    UNION ALL
    SELECT * FROM b2b_sf_non_sql_rev
)



  SELECT
    date,
    CAST(SUM(CASE WHEN line_item_high_level_product = 'eBooks' AND (line_item_type_of_business = "New" OR line_item_type_of_business IS NULL) THEN revenue ELSE 0 END) as float64) AS mqlRevBksNew,
    CAST(SUM(CASE WHEN line_item_high_level_product = 'eBooks' AND line_item_type_of_business = 'Upsell' THEN revenue ELSE 0 END) as float64) AS mqlRevBksUpsell,
  FROM combined_data
  WHERE line_item_high_level_product != 'Other'
  GROUP BY 1


)t4

on t1.date = t4.date


left join
(
  select a.date, sum(a.bookLeads) as bookLeads,sum(a.bookLeadsAcc) as bookLeadsAcc,sum(b.bookCntctSgnMkt) as bookCntctSgnMkt
from
(
  select
  format_date('%Y-%m',SubmitDate) as date,
  count(UniqueSubmissionID) as bookLeads,
  sum(case when regexp_contains(CurrentStatus,'(Submission_to_BFlux_succeeded|In_Discussion|Accepted|Submission_to_MyBookSubmission_succeeded|Submission_to_MyBookSubmission_in_progress)') then 1 else 0 end) as bookLeadsAcc
  from `snproject52c6e272.zz_researcher_insights_and_analytics.MarketingReport_V3`
  where cast(SubmitDate as date) > "2023-12-31"
  group by 1
  order by 1
)a

left join
(
  select
  format_date('%Y-%m',ContractSigned) as date,
  count(ContractSigned) as bookCntctSgnMkt,
  from `snproject52c6e272.zz_researcher_insights_and_analytics.MarketingReport_V3`
  where cast(ContractSigned as date) > "2023-12-31"
  group by 1
  order by 1
)b

on a.date = b.date
group by date
order by date
)t5

on t1.date = t5.date

left join

(
  select
  format_date("%Y-%m",month_start_date) as date,

  IFNULL(sum(mkt_gen_subs), 0.0) as submMktGen,
  IFNULL(sum(mkt_gen_subs_oa), 0.0) as submMktGenOa,
  IFNULL(sum(mkt_infl_1plus), 0.0) as submMktInfl1Plus,
  IFNULL(sum(mkt_infl_1plus_collections), 0.0) as submMktInflCol1Plus,
  IFNULL(sum(mkt_infl_3plus), 0.0) as submMktInfl3Plus,
  IFNULL(sum(mkt_infl_3plus_oa), 0.0) as submMktInfl3PlusOa,
  IFNULL(sum(subs_from_pa), 0.0) as powerAuthors

  from `snproject-cbb9ca25.RM_Submissions_reporting.brm_subs_reporting_2025`
  where month_start_date > "2024-12-31"
  group by 1
  order by 1
  )t6

on t1.date = t6.date;

----------------------------------------------------------------------------------
----------------------------------------------------------------------------------

UPDATE `snproject147bce46.2025_kpi_report.dashboard_b2c_ecom`
SET date = FORMAT_DATE('%Y-%m-%d', PARSE_DATE('%Y-%m', date))
WHERE date > "2023-12-31"
